# Mood-Based Watch Recommendation: Data Preprocessing

This notebook is the first data-preparation stage for the final Mood-Based Watch Recommender. Its job is to turn the raw IMDb movies/shows export into a cleaner, defensible dataset that can support later NLP topic modeling, facet labeling, and backend recommendations.

Final-product purpose:

1. preserve reliable title identifiers such as `tconst`, title, year, type, genres, rating, and votes
2. remove columns that do not help the recommendation product
3. identify missing, weak, or mismatched descriptions before modeling
4. apply verified description fills only when the title identity is clear
5. save a cleaner dataset that later notebooks can enrich and transform into the production NLP catalog

Important accuracy rule: this notebook does not treat a title name alone as enough evidence. Description fixes are applied through small, reviewable batches so the final recommender is not trained on mismatched plot text.


## 1. Import Libraries And Set Paths


In [1]:
import re
from pathlib import Path

import pandas as pd

DATA_PATH = Path("IMDb Movies Shows with Descriptions export 2026-05-21 10-00-48.csv")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

CLEANED_PATH = OUTPUT_DIR / "cleaned_watch_data.csv"
CORRECTED_PATH = OUTPUT_DIR / "cleaned_watch_data_corrected.csv"
MISSING_DESCRIPTIONS_PATH = OUTPUT_DIR / "missing_descriptions.csv"
DESCRIPTION_FILLS_PATH = OUTPUT_DIR / "verified_description_fills.csv"
CORRECTED_DESCRIPTION_SOURCES_PATH = OUTPUT_DIR / "corrected_description_sources.csv"
DESCRIPTION_MISMATCH_NEEDS_REVIEW_PATH = OUTPUT_DIR / "description_mismatch_needs_review.csv"
VERIFIED_DESCRIPTION_OK_SOURCES_PATH = OUTPUT_DIR / "verified_description_ok_sources.csv"


## 2. Load The Dataset

Use `na_values` so IMDb-style `\N` values are treated as missing values by pandas.


In [2]:
df = pd.read_csv(DATA_PATH, na_values=[r"\N", ""], keep_default_na=True)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.head()


Rows: 7850
Columns: 21


,Unnamed: 0,index,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,...,averageRating,numVotes,ordering,title,region,language,types,attributes,isOriginalTitle,Description
0,0,0,tt0102926,movie,The Silence of the Lambs,The Silence of the Lambs,0,1991,NaN,118.0,...,8.6,1473918,50,The Silence of the Lambs,US,en,NaN,NaN,0,"Jodie Foster stars as Clarice Starling, a top ..."
1,1,1,tt0103064,movie,Terminator 2: Judgment Day,Terminator 2: Judgment Day,0,1991,NaN,137.0,...,8.6,1128166,17,Terminator 2: Judgment Day,US,en,dvd,NaN,0,"In this sequel set eleven years after ""The Ter..."
2,2,3,tt0110357,movie,The Lion King,The Lion King,0,1994,NaN,88.0,...,8.5,1090882,18,The Lion King 3D,US,en,NaN,3-D version,0,This Disney animated feature follows the adven...
3,3,4,tt0110912,movie,Pulp Fiction,Pulp Fiction,0,1994,NaN,154.0,...,8.9,2118762,22,Pulp Fiction,US,en,NaN,NaN,0,Vincent Vega (John Travolta) and Jules Winnfie...
4,4,5,tt0111161,movie,The Shawshank Redemption,The Shawshank Redemption,0,1994,NaN,142.0,...,9.3,2759621,2,The Shawshank Redemption,US,en,NaN,NaN,0,Andy Dufresne (Tim Robbins) is sentenced to tw...


## 3. Basic Dataset Audit


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7850 entries, 0 to 7849
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       7850 non-null   int64  
 1   index            7850 non-null   int64  
 2   tconst           7850 non-null   str    
 3   titleType        7850 non-null   str    
 4   primaryTitle     7850 non-null   str    
 5   originalTitle    7850 non-null   str    
 6   isAdult          7850 non-null   int64  
 7   startYear        7850 non-null   int64  
 8   endYear          2161 non-null   float64
 9   runtimeMinutes   7727 non-null   float64
 10  genres           7848 non-null   str    
 11  averageRating    7850 non-null   float64
 12  numVotes         7850 non-null   int64  
 13  ordering         7850 non-null   int64  
 14  title            7850 non-null   str    
 15  region           7850 non-null   str    
 16  language         7850 non-null   str    
 17  types            7480 non

In [4]:
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .rename("missing_count")
      .to_frame()
)

missing_summary[missing_summary["missing_count"] > 0]


,missing_count
attributes,7781
endYear,5689
types,370
Description,188
runtimeMinutes,123
genres,2


## 3A. Fix Missing Genres

The raw dataset uses IMDb's `\N` marker for unknown values. After loading with pandas, those become missing values.

There are only two missing `genres` rows:
- `tt13580840`, `Simon`, 2023: verified as `Drama`.
- `tt4876244`, `Gone`, 2016: no reliable genre source found yet, so it is filled as `Unknown` instead of guessing.

This keeps the genre column non-null for preprocessing while preserving accuracy.


In [5]:
missing_genres = df.loc[df["genres"].isna(), [
    "tconst",
    "titleType",
    "primaryTitle",
    "originalTitle",
    "startYear",
    "genres",
    "Description",
]]

missing_genres


,tconst,titleType,primaryTitle,originalTitle,startYear,genres,Description
99,tt13580840,movie,Simon,Simón,2023,NaN,NaN
262,tt4876244,movie,Gone,Gone,2016,NaN,"Two tenacious lawyers (Joel Klug, Dirk Been) h..."


In [6]:
verified_genre_fills = pd.DataFrame([
    {
        "tconst": "tt13580840",
        "primaryTitle": "Simon",
        "startYear": 2023,
        "genres": "Drama",
        "genre_source": "https://en.wikipedia.org/wiki/Sim%C3%B3n_(2023_film)",
        "genre_note": "Verified source identifies Simón as a drama film.",
    },
    {
        "tconst": "tt4876244",
        "primaryTitle": "Gone",
        "startYear": 2016,
        "genres": "Unknown",
        "genre_source": r"Original dataset has IMDb missing marker \N; no reliable external genre source found yet.",
        "genre_note": "Temporary non-null placeholder. Do not use as a mood genre until verified.",
    },
])

verified_genre_fills


,tconst,primaryTitle,startYear,genres,genre_source,genre_note
0,tt13580840,Simon,2023,Drama,https://en.wikipedia.org/wiki/Sim%C3%B3n_(2023...,Verified source identifies Simón as a drama film.
1,tt4876244,Gone,2016,Unknown,Original dataset has IMDb missing marker \N; n...,Temporary non-null placeholder. Do not use as ...


In [7]:
genre_check = verified_genre_fills.merge(
    df[["tconst", "primaryTitle", "startYear", "genres"]],
    on="tconst",
    how="left",
    suffixes=("_fill", "_dataset"),
    indicator=True,
)

assert genre_check["_merge"].eq("both").all(), "At least one genre-fill tconst was not found in the dataset."
assert genre_check["genres_dataset"].isna().all(), "At least one genre fill would overwrite an existing genre."
assert not verified_genre_fills["tconst"].duplicated().any(), "Duplicate tconst values found in genre fills."

print("Genre fill batch is safe to apply.")
genre_check


Genre fill batch is safe to apply.


,tconst,primaryTitle_fill,startYear_fill,genres_fill,genre_source,genre_note,primaryTitle_dataset,startYear_dataset,genres_dataset,_merge
0,tt13580840,Simon,2023,Drama,https://en.wikipedia.org/wiki/Sim%C3%B3n_(2023...,Verified source identifies Simón as a drama film.,Simon,2023,NaN,both
1,tt4876244,Gone,2016,Unknown,Original dataset has IMDb missing marker \N; n...,Temporary non-null placeholder. Do not use as ...,Gone,2016,NaN,both


In [8]:
genre_map = verified_genre_fills.set_index("tconst")["genres"]
genre_source_map = verified_genre_fills.set_index("tconst")["genre_source"]
genre_note_map = verified_genre_fills.set_index("tconst")["genre_note"]

needs_genre = df["genres"].isna()
can_fill_genre = df["tconst"].isin(genre_map.index)

df.loc[needs_genre & can_fill_genre, "genres"] = df.loc[needs_genre & can_fill_genre, "tconst"].map(genre_map)
df["genre_source"] = df["tconst"].map(genre_source_map)
df["genre_note"] = df["tconst"].map(genre_note_map)

print(f"Genres filled: {(needs_genre & can_fill_genre).sum()}")
print(f"Genres still missing: {df['genres'].isna().sum()}")


Genres filled: 2
Genres still missing: 0


## 4. Audit Missing Descriptions

These rows need special care. We keep `tconst`, title, year, and genre together to avoid confusing different movies or shows with the same name.


In [9]:
description_context_columns = [
    "tconst",
    "titleType",
    "primaryTitle",
    "originalTitle",
    "title",
    "startYear",
    "genres",
    "averageRating",
    "numVotes",
]

missing_descriptions = (
    df.loc[df["Description"].isna(), description_context_columns]
      .copy()
      .reset_index(names="original_row")
)

print(f"Missing descriptions: {len(missing_descriptions)}")
missing_descriptions.head(20)


Missing descriptions: 188


,original_row,tconst,titleType,primaryTitle,originalTitle,title,startYear,genres,averageRating,numVotes
0,47,tt0919370,tvEpisode,Encounter,Sesshoku,Encounter,2006,"Animation,Crime,Drama",8.5,4136
1,50,tt0947906,tvEpisode,Wager,Kake,Wager,2007,"Animation,Crime,Drama",8.8,4072
2,52,tt0950513,tvEpisode,Execution,Shikkou,Execution,2007,"Animation,Crime,Drama",8.0,3748
3,54,tt0990612,tvEpisode,Revival,Fukkatsu,Revival,2007,"Animation,Crime,Drama",9.2,5375
4,82,tt11866324,movie,Prey,Prey,Prey,2022,"Action,Adventure,Drama",7.1,208923
5,84,tt11991748,movie,Rurouni Kenshin: Final Chapter Part II - The B...,Rurôni Kenshin: Sai shûshô - The Beginning,Rurouni Kenshin: Final Chapter Part II - The B...,2021,"Action,Adventure,Drama",7.4,9737
6,99,tt13580840,movie,Simon,Simón,Simon,2023,Drama,9.4,14
7,145,tt1650062,movie,Super 8,Super 8,Super 8,2011,"Action,Mystery,Sci-Fi",7.0,361538
8,153,tt1825683,movie,Black Panther,Black Panther,Black Panther,2018,"Action,Adventure,Sci-Fi",7.3,805648
9,264,tt4935462,movie,Amazing Grace,Amazing Grace,Amazing Grace,2018,"Documentary,Music",7.5,4919


In [10]:
MISSING_DESCRIPTIONS_PATH


PosixPath('data/missing_descriptions.csv')

## 5. Remove Columns That Are Not Useful For The Recommender

`runtimeMinutes` is intentionally dropped because runtime is not part of the current recommendation logic.

The unnamed column, `index`, and `ordering` are also technical/import columns rather than useful recommendation features.


In [11]:
columns_to_drop = ["Unnamed: 0", "index", "ordering", "runtimeMinutes"]
columns_to_drop = [column for column in columns_to_drop if column in df.columns]

watch_df = df.drop(columns=columns_to_drop).copy()
watch_df.columns.tolist()


['tconst',
 'titleType',
 'primaryTitle',
 'originalTitle',
 'isAdult',
 'startYear',
 'endYear',
 'genres',
 'averageRating',
 'numVotes',
 'title',
 'region',
 'language',
 'types',
 'attributes',
 'isOriginalTitle',
 'Description',
 'genre_source',
 'genre_note']

## 6. Load Verified Description Fills

The file `data/verified_description_fills.csv` contains sourced descriptions keyed by `tconst`.

This is kept as a CSV instead of a huge notebook cell so the notebook stays readable. Each row includes:
- `tconst`
- `primaryTitle`
- `startYear`
- `genres`
- `description`
- `source`

Important: `Cheat` (2019, `tt9131136`) is intentionally not filled yet because no reliable exact source was found. Accuracy is more important than forcing a guess.


In [12]:
verified_description_fills = pd.read_csv(DESCRIPTION_FILLS_PATH)

print(f"Verified description fills loaded: {len(verified_description_fills)}")
verified_description_fills.head(20)


Verified description fills loaded: 187


,tconst,primaryTitle,startYear,genres,description,source
0,tt11866324,Prey,2022.0,"Action,Adventure,Drama","In 1719 on the Great Plains, Naru, a skilled y...",https://en.wikipedia.org/wiki/Prey_(2022_film)
1,tt11991748,Rurouni Kenshin: Final Chapter Part II - The B...,2021.0,"Action,Adventure,Drama","In Bakumatsu-era Kyoto, Himura Kenshin serves ...",https://en.wikipedia.org/wiki/Rurouni_Kenshin:...
2,tt13580840,Simon,2023.0,Drama,After being arrested and tortured during Venez...,https://en.wikipedia.org/wiki/Sim%C3%B3n_(2023...
3,tt1650062,Super 8,2011.0,"Action,Mystery,Sci-Fi","In 1979 Ohio, a group of children making a hom...",https://en.wikipedia.org/wiki/Super_8_(2011_film)
4,tt1825683,Black Panther,2018.0,"Action,Adventure,Sci-Fi","After his father’s death, T’Challa returns to ...",https://en.wikipedia.org/wiki/Black_Panther_(f...
5,tt4935462,Amazing Grace,2018.0,"Documentary,Music",This concert documentary captures Aretha Frank...,https://en.wikipedia.org/wiki/Amazing_Grace_(2...
6,tt9626278,Fourteen,2019.0,Drama,"Over the course of a decade, two longtime frie...",https://en.wikipedia.org/wiki/Fourteen_(film)
7,tt0112047,Life with Louie,1994.0,"Adventure,Animation,Comedy",Inspired by comedian Louie Anderson’s childhoo...,https://en.wikipedia.org/wiki/Life_with_Louie
8,tt0919370,Encounter,2006.0,"Animation,Crime,Drama",Light enters college and meets an eccentric st...,https://en.wikipedia.org/wiki/Death_Note_(2006...
9,tt0947906,Wager,2007.0,"Animation,Crime,Drama","L warns that if he dies, Light should be arres...",https://en.wikipedia.org/wiki/Death_Note_(2006...


## 7. Validate The Verified Fill Batch Before Applying It

This checks that every fill row matches exactly one dataset row by `tconst`, then compares title/year/genre context before writing anything into `watch_df`.


In [13]:
fill_check = verified_description_fills.merge(
    df[["tconst", "primaryTitle", "startYear", "genres", "Description"]],
    on="tconst",
    how="left",
    suffixes=("_fill", "_dataset"),
    indicator=True,
)

fill_check[[
    "tconst",
    "primaryTitle_fill",
    "primaryTitle_dataset",
    "startYear_fill",
    "startYear_dataset",
    "genres_fill",
    "genres_dataset",
    "Description",
    "_merge",
]]


,tconst,primaryTitle_fill,primaryTitle_dataset,startYear_fill,startYear_dataset,genres_fill,genres_dataset,Description,_merge
0,tt11866324,Prey,Prey,2022.0,2022,"Action,Adventure,Drama","Action,Adventure,Drama",NaN,both
1,tt11991748,Rurouni Kenshin: Final Chapter Part II - The B...,Rurouni Kenshin: Final Chapter Part II - The B...,2021.0,2021,"Action,Adventure,Drama","Action,Adventure,Drama",NaN,both
2,tt13580840,Simon,Simon,2023.0,2023,Drama,Drama,NaN,both
3,tt1650062,Super 8,Super 8,2011.0,2011,"Action,Mystery,Sci-Fi","Action,Mystery,Sci-Fi",NaN,both
4,tt1825683,Black Panther,Black Panther,2018.0,2018,"Action,Adventure,Sci-Fi","Action,Adventure,Sci-Fi",NaN,both
...,...,...,...,...,...,...,...,...,...
182,tt7843600,Fyre Fraud,Fyre Fraud,2019.0,2019,Documentary,Documentary,NaN,both
183,tt8015444,Last and First Men,Last and First Men,2020.0,2020,"Fantasy,Mystery,Sci-Fi","Fantasy,Mystery,Sci-Fi",NaN,both
184,tt8359842,It Must Be Heaven,It Must Be Heaven,2019.0,2019,"Comedy,Drama","Comedy,Drama",NaN,both
185,tt9118930,Wild Bill,Wild Bill,2019.0,2019,"Comedy,Crime,Drama","Comedy,Crime,Drama",NaN,both


In [14]:
assert fill_check["_merge"].eq("both").all(), "At least one tconst was not found in the dataset."
assert fill_check["Description"].isna().all(), "At least one fill is trying to overwrite an existing description."
assert not verified_description_fills["tconst"].duplicated().any(), "Duplicate tconst values found in fill batch."

print("Fill batch is safe to apply.")


Fill batch is safe to apply.


## 8. Apply Verified Descriptions And Drop Unusable Rows

After verified descriptions are applied, rows that still have no usable description are removed from the recommender-ready dataset.

Reason: the final product depends on description text for topic modeling, mood/facet labeling, prompt matching, and explanation quality. Keeping titles with no verified story text would make the recommendation catalog look larger but less trustworthy.


In [15]:
description_map = verified_description_fills.set_index("tconst")["description"]
source_map = verified_description_fills.set_index("tconst")["source"]

needs_description = watch_df["Description"].isna()
can_fill = watch_df["tconst"].isin(description_map.index)

watch_df.loc[needs_description & can_fill, "Description"] = watch_df.loc[needs_description & can_fill, "tconst"].map(description_map)
watch_df["description_source"] = watch_df["tconst"].map(source_map)

rows_still_missing_description = watch_df.loc[
    watch_df["Description"].isna(),
    ["tconst", "primaryTitle", "startYear", "genres", "Description"],
].copy()

print(f"Descriptions filled in this batch: {(needs_description & can_fill).sum()}")
print(f"Rows still missing descriptions before drop: {len(rows_still_missing_description)}")

rows_still_missing_description


Descriptions filled in this batch: 187
Rows still missing descriptions before drop: 1


,tconst,primaryTitle,startYear,genres,Description
5807,tt9131136,Cheat,2019,Thriller,NaN


## 9. Build Final Recommender Dataset And Save

The exported `cleaned_watch_data.csv` should only contain columns that help the recommender or future modeling.

Dropped from the final CSV:
- mostly missing metadata: `endYear`, `types`, `attributes`
- constant columns: `isAdult`, `language`, `isOriginalTitle`
- import/title bookkeeping: `index`, `ordering`, unnamed column, `region`
- audit-only columns: `genre_source`, `genre_note`, `description_source`

Kept or created:
- stable ID: `tconst`
- title fields for display/search: `primary_title`, `display_title`, `original_title`
- recommender features: `content_type`, `release_year`, `genres`, `description`
- ranking features: `average_rating`, `num_votes`, `weighted_rating`
- text modeling feature: `combined_features`


In [16]:
dropped_missing_descriptions = rows_still_missing_description.copy()

watch_df = watch_df.dropna(subset=["Description"]).copy()

# One duplicated IMDb title ID exists because the original file has the same title under
# different title-region metadata. Since region is not a recommender feature, keep one row.
duplicate_tconst_rows = watch_df.loc[watch_df["tconst"].duplicated(keep=False)].copy()
watch_df = watch_df.drop_duplicates(subset=["tconst"], keep="first").copy()

final_df = pd.DataFrame({
    "tconst": watch_df["tconst"].astype(str),
    "content_type": watch_df["titleType"].astype(str),
    "primary_title": watch_df["primaryTitle"].astype(str),
    "display_title": watch_df["title"].astype(str),
    "original_title": watch_df["originalTitle"].astype(str),
    "release_year": watch_df["startYear"].astype(int),
    "genres": watch_df["genres"].astype(str),
    "average_rating": watch_df["averageRating"].astype(float).round(1),
    "num_votes": watch_df["numVotes"].astype(int),
    "description": watch_df["Description"].astype(str).str.strip(),
})

# IMDb-style weighted rating gives a more reliable ranking signal than raw rating alone.
# It reduces the chance of tiny-vote titles dominating recommendations.
minimum_votes = final_df["num_votes"].quantile(0.75)
mean_rating = final_df["average_rating"].mean()
final_df["weighted_rating"] = (
    (final_df["num_votes"] / (final_df["num_votes"] + minimum_votes)) * final_df["average_rating"]
    + (minimum_votes / (final_df["num_votes"] + minimum_votes)) * mean_rating
).round(3)

final_df["combined_features"] = (
    final_df["primary_title"] + " "
    + final_df["display_title"] + " "
    + final_df["original_title"] + " "
    + final_df["content_type"] + " "
    + final_df["genres"].str.replace(",", " ", regex=False) + " "
    + final_df["description"]
).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

remaining_missing_descriptions = (
    final_df.loc[final_df["description"].isna() | final_df["description"].eq(""), [
        "tconst",
        "primary_title",
        "release_year",
        "genres",
        "description",
    ]]
    .copy()
    .reset_index(names="original_row")
)

final_df.to_csv(CLEANED_PATH, index=False)
watch_df = final_df.copy()

print(f"Rows dropped because Description could not be verified: {len(dropped_missing_descriptions)}")
print(f"Duplicate tconst rows removed: {len(duplicate_tconst_rows) - duplicate_tconst_rows['tconst'].nunique()}")
print(f"Final cleaned rows: {len(final_df)}")
print(f"Final cleaned columns: {len(final_df.columns)}")
print(f"Cleaned dataset saved to: {CLEANED_PATH}")
print(f"Remaining missing descriptions: {len(remaining_missing_descriptions)}")

final_df.head()


Rows dropped because Description could not be verified: 1
Duplicate tconst rows removed: 1
Final cleaned rows: 7848
Final cleaned columns: 12
Cleaned dataset saved to: data/cleaned_watch_data.csv
Remaining missing descriptions: 0


,tconst,content_type,primary_title,display_title,original_title,release_year,genres,average_rating,num_votes,description,weighted_rating,combined_features
0,tt0102926,movie,The Silence of the Lambs,The Silence of the Lambs,The Silence of the Lambs,1991,"Crime,Drama,Thriller",8.6,1473918,"Jodie Foster stars as Clarice Starling, a top ...",8.554,the silence of the lambs the silence of the la...
1,tt0103064,movie,Terminator 2: Judgment Day,Terminator 2: Judgment Day,Terminator 2: Judgment Day,1991,"Action,Sci-Fi",8.6,1128166,"In this sequel set eleven years after ""The Ter...",8.541,terminator 2: judgment day terminator 2: judgm...
2,tt0110357,movie,The Lion King,The Lion King 3D,The Lion King,1994,"Adventure,Animation,Drama",8.5,1090882,This Disney animated feature follows the adven...,8.444,the lion king the lion king 3d the lion king m...
3,tt0110912,movie,Pulp Fiction,Pulp Fiction,Pulp Fiction,1994,"Crime,Drama",8.9,2118762,Vincent Vega (John Travolta) and Jules Winnfie...,8.860,pulp fiction pulp fiction pulp fiction movie c...
4,tt0111161,movie,The Shawshank Redemption,The Shawshank Redemption,The Shawshank Redemption,1994,Drama,9.3,2759621,Andy Dufresne (Tim Robbins) is sentenced to tw...,9.261,the shawshank redemption the shawshank redempt...


## 10. Audit And Correct Description Mismatches

These corrections handle externally verified title-description mismatches keyed by `tconst`. The original `cleaned_watch_data.csv` is left untouched; the corrected dataset and audit files are saved separately.


In [17]:
DESCRIPTION_MISMATCH_CORRECTIONS = {'tt7286456': {'description': 'In grim 1980s Gotham, isolated party clown Arthur Fleck struggles '
                              'with poverty, illness, and public humiliation until his search for '
                              'dignity curdles into violence and a chaotic criminal persona.',
               'source_url': 'https://en.wikipedia.org/wiki/Joker_(2019_film)',
               'note': 'Verified by tconst/title/year: Joker (2019), Todd Phillips psychological '
                       'thriller; old description names Yuri Kuzmenko and unrelated actors.'},
 'tt6751668': {'description': 'A poor Seoul family schemes its way into jobs inside a wealthy '
                              'household, but the class satire turns darker as hidden debts, envy, '
                              'and buried secrets erupt into violence.',
               'source_url': 'https://en.wikipedia.org/wiki/Parasite_(2019_film)',
               'note': 'Verified by tconst/title/year/original title Gisaengchung; old '
                       'paranormal-scientist description belongs to a different Parasite.'},
 'tt1663202': {'description': 'After a bear mauling leaves frontiersman Hugh Glass near death in '
                              'the 1820s wilderness, betrayal and grief drive him through a brutal '
                              'survival journey toward revenge.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Revenant_(2015_film)',
               'note': 'Verified by tconst/title/year and western survival plot; old '
                       'undead-soldier description belongs to another Revenant.'},
 'tt1392214': {'description': 'When two young girls vanish in Pennsylvania, a desperate father '
                              'crosses moral lines while a relentless detective follows a maze of '
                              'clues in a bleak kidnapping thriller.',
               'source_url': 'https://en.wikipedia.org/wiki/Prisoners_(2013_film)',
               'note': 'Verified by tconst/title/year and Keller Dover/Detective Loki plot; old '
                       'Vietnam POW description belongs to another Prisoners.'},
 'tt2096673': {'description': 'Inside the mind of young Riley, Joy, Sadness, Anger, Fear, and '
                              'Disgust struggle to guide her through a painful move, turning a '
                              'bright coming-of-age story into a tender lesson about emotional '
                              'balance.',
               'source_url': 'https://en.wikipedia.org/wiki/Inside_Out',
               'note': 'Verified by tconst/title/year/Pixar plot; old Steven Weber neighbor '
                       'mystery belongs to another Inside Out.'},
 'tt2543164': {'description': 'A linguist is recruited after alien vessels appear across Earth, '
                              'racing to decode their language before fear and military tension '
                              'push humanity toward catastrophe.',
               'source_url': 'https://en.wikipedia.org/wiki/Arrival_(film)',
               'note': 'Verified by tconst/title/year and Denis Villeneuve alien-contact premise; '
                       'old FBI serial-killer description belongs to another Arrival.'},
 'tt0118799': {'description': 'In fascist Italy, warm-hearted Guido turns romance, humor, and '
                              'imagination into acts of protection when he and his young son are '
                              'imprisoned in a Nazi concentration camp.',
               'source_url': 'https://en.wikipedia.org/wiki/Life_Is_Beautiful',
               'note': 'Verified by tconst/title/year/original title La vita e bella; old '
                       'Angolan-pilot description belongs to the 1979 same-title film.'},
 'tt0253474': {'description': 'Polish Jewish pianist Wladyslaw Szpilman survives the Nazi '
                              'occupation of Warsaw through loss, hiding, hunger, and fragile acts '
                              'of mercy in a harrowing Holocaust drama.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Pianist_(2002_film)',
               'note': 'Verified by tconst/title/year and Wladyslaw Szpilman plot; old '
                       'Canadian-sisters description belongs to The Pianist (1991).'},
 'tt0112573': {'description': 'Scottish warrior William Wallace leads a bloody rebellion against '
                              'English rule after personal tragedy, blending epic battles, '
                              'martyrdom, and nationalist defiance.',
               'source_url': 'https://en.wikipedia.org/wiki/Braveheart',
               'note': 'Verified by tconst/title/year and William Wallace premise; old Standing '
                       'Rock fishing-rights description belongs to another Braveheart.'},
 'tt0113277': {'description': 'A disciplined Los Angeles master thief and an obsessive detective '
                              'circle each other through robberies, betrayals, and personal '
                              'collapse in a tense crime epic.',
               'source_url': 'https://en.wikipedia.org/wiki/Heat_(1995_film)',
               'note': 'Verified by tconst/title/year and Los Angeles crime plot; old high-school '
                       'reunion description belongs to another Heat.'},
 'tt0451279': {'description': 'Amazon warrior Diana leaves Themyscira during World War I with '
                              'pilot Steve Trevor, believing she can stop the god of war and '
                              'discovering both human cruelty and compassion.',
               'source_url': 'https://en.wikipedia.org/wiki/Wonder_Woman_(2017_film)',
               'note': 'Verified by tconst/title/year and DCEU World War I plot; old Cathy Lee '
                       'Crosby CIA description belongs to the 1974 television film.'},
 'tt0780504': {'description': 'A quiet Los Angeles stunt driver moonlights as a getaway '
                              'specialist, then risks everything to protect his neighbor when a '
                              'robbery pulls them into mob violence.',
               'source_url': 'https://en.wikipedia.org/wiki/Drive_(2011_film)',
               'note': 'Verified by tconst/title/year and Ryan Gosling Driver plot; old bionic-man '
                       'description belongs to another Drive.'},
 'tt2294629': {'description': "After Queen Elsa's hidden ice magic traps Arendelle in winter, her "
                              'sister Anna sets out with unlikely companions to heal their bond '
                              'and save the kingdom.',
               'source_url': 'https://en.wikipedia.org/wiki/Frozen_(2013_film)',
               'note': 'Verified by tconst/title/year/Disney animation; old performance-artist '
                       'suicide description belongs to another Frozen.'},
 'tt1637725': {'description': "A childhood wish brings John Bennett's teddy bear to life, but "
                              'adulthood forces him to choose between a chaotic lifelong '
                              'friendship and a more mature future with his girlfriend.',
               'source_url': 'https://en.wikipedia.org/wiki/Ted_(film)',
               'note': 'Verified by tconst/title/year and Seth MacFarlane comedy premise; old '
                       'Unabomber-style description belongs to another Ted.'},
 'tt1190634': {'description': 'In a world where superheroes are corporate celebrities, a vigilante '
                              'group exposes corruption, abuse, and violence behind the polished '
                              'public image of the powerful Seven.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Boys_(TV_series)',
               'note': 'Verified by tconst/title/year/content type and Amazon superhero satire; '
                       'old novelist-small-town description belongs to another The Boys.'},
 'tt11286314': {'description': 'Two astronomers discover a planet-killing comet and try to warn an '
                               'image-obsessed America, turning disaster-movie stakes into furious '
                               'political and media satire.',
                'source_url': 'https://en.wikipedia.org/wiki/Don%27t_Look_Up',
                'note': 'Verified by tconst/title/year and Adam McKay comet satire; old '
                        "Transylvania film-crew description belongs to another Don't Look Up."},
 'tt2380307': {'description': 'Music-loving Miguel is transported to the Land of the Dead on Dia '
                              'de Muertos, where he uncovers family secrets and learns how memory, '
                              'grief, and art keep loved ones alive.',
               'source_url': 'https://en.wikipedia.org/wiki/Coco_(2017_film)',
               'note': 'Verified by tconst/title/year/Pixar plot; old bar-mitzvah party '
                       'description belongs to another Coco.'},
 'tt1483013': {'description': 'On a ruined future Earth, drone technician Jack Harper questions '
                              'his mission after rescuing a mysterious survivor whose memories '
                              'expose the truth about an alien war.',
               'source_url': 'https://en.wikipedia.org/wiki/Oblivion_(2013_film)',
               'note': "Verified by tconst/title/year/Tom Cruise sci-fi premise; old sheriff's-son "
                       'western description belongs to another Oblivion.'},
 'tt0112641': {'description': 'A mob-backed casino operator, a volatile enforcer, and a magnetic '
                              'hustler chase power and money in Las Vegas as greed, addiction, and '
                              'surveillance pull their empire apart.',
               'source_url': 'https://en.wikipedia.org/wiki/Casino_(1995_film)',
               'note': 'Verified by tconst/title/year and Scorsese Las Vegas crime plot; old '
                       'floating-casino description belongs to another Casino.'},
 'tt4972582': {'description': 'Three teenage girls are abducted by Kevin, a man with dissociative '
                              'identities, as one captive tries to survive the arrival of a '
                              'violent emerging persona.',
               'source_url': 'https://en.wikipedia.org/wiki/Split_(2016_American_film)',
               'note': 'Verified by tconst/title/year and M. Night Shyamalan kidnapping thriller; '
                       'old bowling-romance description belongs to another Split.'},
 'tt2713180': {'description': 'In the final weeks of World War II, a hardened Sherman tank crew '
                              'and a young replacement soldier face exhaustion, terror, and '
                              'loyalty during a last stand in Germany.',
               'source_url': 'https://en.wikipedia.org/wiki/Fury_(2014_film)',
               'note': 'Verified by tconst/title/year and WWII tank-crew plot; old '
                       'haunted-building musician description belongs to another Fury.'},
 'tt1895587': {'description': "The Boston Globe's investigative team uncovers systemic child abuse "
                              'and institutional protection inside the Catholic Church, building a '
                              'restrained drama about journalism and accountability.',
               'source_url': 'https://en.wikipedia.org/wiki/Spotlight_(film)',
               'note': 'Verified by tconst/title/year and Boston Globe investigation; old '
                       'music/light-show description belongs to another Spotlight.'},
 'tt1291584': {'description': 'Estranged brothers enter the same high-stakes MMA tournament, '
                              'forcing their broken family to confront addiction, resentment, '
                              'duty, and the need for forgiveness.',
               'source_url': 'https://en.wikipedia.org/wiki/Warrior_(2011_film)',
               'note': 'Verified by tconst/title/year and MMA brothers premise; old disabled Army '
                       'veteran description belongs to another Warrior.'},
 'tt2379713': {'description': 'James Bond follows a posthumous clue from M into the secret '
                              'organization Spectre while MI6 faces a political threat from a '
                              'global surveillance program.',
               'source_url': 'https://en.wikipedia.org/wiki/Spectre_(2015_film)',
               'note': 'Verified by tconst/title/year/Bond plot; old haunted Irish mansion '
                       'description belongs to another Spectre.'},
 'tt3397884': {'description': 'An idealistic FBI agent joins a covert task force targeting a '
                              'Mexican cartel, entering a morally murky border war led by '
                              'operatives with hidden agendas.',
               'source_url': 'https://en.wikipedia.org/wiki/Sicario_(2015_film)',
               'note': 'Verified by tconst/title/year and Denis Villeneuve cartel-thriller '
                       'premise; old teenage-assassin description belongs to another Sicario.'},
 'tt2382320': {'description': 'Retired James Bond is drawn back into espionage after a stolen '
                              'DNA-targeting bioweapon leads him to old betrayals, new allies, and '
                              'a final confrontation with Safin.',
               'source_url': 'https://en.wikipedia.org/wiki/No_Time_to_Die',
               'note': 'Verified by tconst/title/year and Bond bioweapon plot; old Ghana '
                       'hearse-driver romance description belongs to another No Time to Die.'},
 'tt4954522': {'description': 'A lifelong vegetarian begins veterinary school, tastes meat during '
                              'a hazing ritual, and spirals into disturbing cravings in a visceral '
                              'coming-of-age body-horror story.',
               'source_url': 'https://en.wikipedia.org/wiki/Raw_(film)',
               'note': 'Verified by tconst/title/year/original title Grave; old surfing '
                       'description belongs to another Raw.'},
 'tt1486834': {'description': 'A cynical medical-school dropout forms a close friendship with an '
                              'attached artist, testing whether romantic chemistry can stay '
                              'platonic in a bittersweet modern comedy.',
               'source_url': 'https://en.wikipedia.org/wiki/The_F_Word_(2013_film)',
               'note': 'Verified by tconst/title/year and alternate title The F Word; old '
                       'schizophrenia description belongs to another What If.'},
 'tt0338095': {'description': 'Two friends arrive at a secluded family farmhouse, where a savage '
                              'home invasion turns into a relentless French slasher about fear, '
                              'obsession, and survival.',
               'source_url': 'https://en.wikipedia.org/wiki/High_Tension',
               'note': 'Verified by tconst/title/year/original title Haute tension; old '
                       'deep-sea-diver romance description belongs to another High Tension.'},
 'tt3829920': {'description': 'An elite Arizona firefighting crew fights for Hotshot certification '
                              'and battles deadly wildfires, culminating in a tragic test of duty, '
                              'brotherhood, and sacrifice.',
               'source_url': 'https://en.wikipedia.org/wiki/Only_the_Brave_(2017_film)',
               'note': 'Verified by tconst/title/year and Granite Mountain Hotshots plot; old '
                       'Australian teen friendship description belongs to another Only the Brave.'},
 'tt4873118': {'description': 'After an Afghan interpreter saves a wounded U.S. soldier from a '
                              'Taliban ambush, the soldier returns to repay the debt and rescue '
                              'him from danger.',
               'source_url': 'https://en.wikipedia.org/wiki/Guy_Ritchie%27s_The_Covenant',
               'note': "Verified by tconst/title/year/display title Guy Ritchie's The Covenant; "
                       'old Nazi-gold banker description belongs to another Covenant.'},
 'tt7282468': {'description': 'A lonely aspiring writer reconnects with a childhood acquaintance '
                              'and becomes obsessed with the wealthy, enigmatic man she brings '
                              'back from abroad after she vanishes.',
               'source_url': 'https://en.wikipedia.org/wiki/Burning_(2018_film)',
               'note': 'Verified by tconst/title/year/original title Beoning; old bulimia '
                       'description belongs to another Burning.'},
 'tt1937149': {'description': "When a woman's body disappears from a morgue, her widower and a "
                              'police inspector unravel an unnerving mystery of infidelity, '
                              'murder, and possible revenge.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Body_(2012_film)',
               'note': 'Verified by tconst/title/year/original title El cuerpo; old hit-man '
                       'description belongs to another The Body.'},
 'tt4901306': {'description': 'During a dinner-party eclipse, seven friends agree to share every '
                              'phone call and message, exposing secrets that threaten marriages, '
                              'friendships, and self-image.',
               'source_url': 'https://en.wikipedia.org/wiki/Perfect_Strangers_(2016_film)',
               'note': 'Verified by tconst/title/year/original title Perfetti sconosciuti; old '
                       'jury-duty romance description belongs to another Perfect Strangers.'},
 'tt0443536': {'description': 'Little Red Riding Hood becomes the center of a fractured fairy-tale '
                              'investigation as police compare competing stories from Red, Granny, '
                              'the Wolf, and the Woodsman.',
               'source_url': 'https://en.wikipedia.org/wiki/Hoodwinked!',
               'note': 'Verified by tconst/title/year/original title Hoodwinked!; old bank-robber '
                       'western description belongs to another Hoodwinked.'},
 'tt5186714': {'description': 'A Tehran couple performing Death of a Salesman is shaken after the '
                              'wife is assaulted, sending her husband into a tense search for the '
                              'attacker and a moral reckoning.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Salesman_(2016_film)',
               'note': 'Verified by tconst/title/year/original title Forushande; old Alan Rickman '
                       'affair description belongs to another Salesman.'},
 'tt1979320': {'description': 'Formula One rivals James Hunt and Niki Lauda push each other '
                              'through ambition, danger, and near-fatal competition during the '
                              'dramatic 1976 racing season.',
               'source_url': 'https://en.wikipedia.org/wiki/Rush_(2013_film)',
               'note': 'Verified by tconst/title/year and Hunt-Lauda Formula One premise; old '
                       'post-apocalyptic road-warrior description belongs to another Rush.'},
 'tt1535108': {'description': 'In 2154, a dying factory worker on a ruined Earth fights to reach '
                              'the wealthy orbital habitat Elysium, where medical technology can '
                              'save him and expose a brutal class divide.',
               'source_url': 'https://en.wikipedia.org/wiki/Elysium_(film)',
               'note': 'Verified by tconst/title/year and Neill Blomkamp dystopian sci-fi premise; '
                       'old runaway/homeless-immigrant description belongs to another Elysium.'},
 'tt3480822': {'description': 'After the events of Civil War, Natasha Romanoff confronts her past, '
                              'reunites with her former surrogate family, and battles the Red Room '
                              'that shaped her as an assassin.',
               'source_url': 'https://en.wikipedia.org/wiki/Black_Widow_(2021_film)',
               'note': 'Verified by tconst/title/year and Marvel Natasha Romanoff plot; old '
                       'revenge-on-cheating-men description belongs to another Black Widow.'},
 'tt0230600': {'description': 'In a secluded, fogbound mansion after World War II, a devout mother '
                              'protects her light-sensitive children while increasingly eerie '
                              'events challenge her sense of reality.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Others_(2001_film)',
               'note': 'Verified by tconst/title/year and Nicole Kidman gothic ghost-story '
                       'premise; old librarian-son-suicide description belongs to another The '
                       'Others.'},
 'tt1302006': {'description': 'Aging hitman Frank Sheeran looks back on his years with Russell '
                              "Bufalino's crime family and his fraught loyalty to Teamsters leader "
                              'Jimmy Hoffa.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Irishman',
               'note': 'Verified by tconst/title/year and Scorsese/Sheeran-Hoffa plot; old Danny '
                       'Greene turf-war description belongs to Kill the Irishman.'},
 'tt0375679': {'description': 'Across Los Angeles, police officers, criminals, politicians, and '
                              'families collide in interlocking stories about racism, fear, guilt, '
                              'and fragile attempts at connection.',
               'source_url': 'https://en.wikipedia.org/wiki/Crash_(2004_film)',
               'note': 'Verified by tconst/title/year and Paul Haggis ensemble Los Angeles '
                       'race-relations plot; old newborn/accident description belongs to another '
                       'Crash.'},
 'tt1907668': {'description': 'After miraculously crash-landing a failing airliner, pilot Whip '
                              'Whitaker is hailed as a hero while an investigation exposes '
                              'addiction, denial, and moral reckoning.',
               'source_url': 'https://en.wikipedia.org/wiki/Flight_(2012_film)',
               'note': 'Verified by tconst/title/year and Denzel Washington airline-crash premise; '
                       'old Jamaican boy moon-dream description belongs to another Flight.'},
 'tt3521164': {'description': "A restless Polynesian chief's daughter sails beyond the reef with "
                              "demigod Maui to restore Te Fiti's heart and save her island from "
                              'ecological decay.',
               'source_url': 'https://en.wikipedia.org/wiki/Moana_(2016_film)',
               'note': 'Verified by tconst/title/year and Disney Polynesian voyage premise; old '
                       'Robert Flaherty Samoa documentary description belongs to the 1926 Moana.'},
 'tt2106476': {'description': 'A kind kindergarten teacher in a small Danish town becomes the '
                              "target of mass hysteria after a child's false accusation destroys "
                              'his work, friendships, and safety.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Hunt_(2012_film)',
               'note': 'Verified by tconst/title/year/original title Jagten and Lucas '
                       'false-accusation plot; old Spanish Civil War hunting description belongs '
                       'to another The Hunt.'},
 'tt1306980': {'description': 'A young radio producer diagnosed with spinal cancer leans on humor, '
                              'friendship, therapy, and family tension while trying to survive '
                              'fear and uncertainty.',
               'source_url': 'https://en.wikipedia.org/wiki/50/50_(2011_film)',
               'note': 'Verified by tconst/title/year and cancer dramedy premise; old '
                       'gangster-bungalow robbery description belongs to another 50/50.'},
 'tt4975722': {'description': 'Across three stages of life, Chiron grows up in Miami while '
                              'struggling with identity, masculinity, longing, and the tenderness '
                              'he rarely receives.',
               'source_url': 'https://en.wikipedia.org/wiki/Moonlight_(2016_film)',
               'note': 'Verified by tconst/title/year and Chiron coming-of-age structure; old '
                       'publishing-job description belongs to another Moonlight.'},
 'tt6857112': {'description': "A family's beach vacation turns into a nightmare when their "
                              'red-clad doubles appear, forcing them into a violent confrontation '
                              'with a hidden national underclass.',
               'source_url': 'https://en.wikipedia.org/wiki/Us_(2019_film)',
               'note': 'Verified by tconst/title/year and Jordan Peele doppelganger horror '
                       'premise; old Swedish teachers affair description belongs to another Us.'},
 'tt1152836': {'description': 'During the Great Depression, bank robber John Dillinger becomes a '
                              'celebrity outlaw while FBI agent Melvin Purvis leads an escalating '
                              'manhunt against him.',
               'source_url': 'https://en.wikipedia.org/wiki/Public_Enemies_(2009_film)',
               'note': 'Verified by tconst/title/year and John Dillinger/Melvin Purvis plot; old '
                       'Ma Barker description belongs to another Public Enemies.'},
 'tt1598778': {'description': 'A lethal virus spreads globally as scientists, public health '
                              'officials, families, and opportunists struggle with fear, '
                              'misinformation, grief, and containment.',
               'source_url': 'https://en.wikipedia.org/wiki/Contagion_(2011_film)',
               'note': 'Verified by tconst/title/year and Steven Soderbergh pandemic plot; old '
                       'fantasy-mansion murder description belongs to another Contagion.'},
 'tt1478338': {'description': 'A broke and insecure woman tries to serve as maid of honor for her '
                              'best friend, spiraling through rivalry, humiliation, and messy '
                              'loyalty before the wedding.',
               'source_url': 'https://en.wikipedia.org/wiki/Bridesmaids_(2011_film)',
               'note': 'Verified by tconst/title/year and Annie/Maya wedding-party comedy premise; '
                       'old hometown reunion wedding description belongs to another Bridesmaids.'},
 'tt0203009': {'description': 'In bohemian Paris, a young writer falls for Moulin Rouge star '
                              'Satine, but their romance is threatened by illness, jealousy, and '
                              'the demands of a powerful patron.',
               'source_url': 'https://en.wikipedia.org/wiki/Moulin_Rouge!',
               'note': 'Verified by tconst/title/year and Baz Luhrmann musical romance; old '
                       'aristocrat/mother romance description belongs to another Moulin Rouge.'},
 'tt7888964': {'description': 'A mild-mannered family man with a hidden past snaps after a home '
                              "invasion and becomes the target of a Russian crime lord's revenge.",
               'source_url': 'https://en.wikipedia.org/wiki/Nobody_(2021_film)',
               'note': 'Verified by tconst/title/year and Hutch Mansell action-thriller premise; '
                       'old morgue-worker description belongs to another Nobody.'},
 'tt4160708': {'description': "Three young burglars break into a blind veteran's isolated house, "
                              'only to discover that their intended victim is dangerous and hiding '
                              'horrifying secrets.',
               'source_url': 'https://en.wikipedia.org/wiki/Don%27t_Breathe',
               'note': 'Verified by tconst/title/year and blind-veteran home-invasion premise; old '
                       "Tbilisi prognosis description belongs to another Don't Breathe."},
 'tt1922777': {'description': 'A true-crime writer moves his family into a murder house and '
                              'discovers disturbing films that connect a string of killings to a '
                              'malevolent supernatural force.',
               'source_url': 'https://en.wikipedia.org/wiki/Sinister_(film)',
               'note': 'Verified by tconst/title/year and Ellison Oswalt/Bughuul horror premise; '
                       'old conjurer description belongs to another Sinister.'},
 'tt5362988': {'description': "On Wyoming's Wind River reservation, a wildlife tracker and an FBI "
                              "agent investigate a young woman's death in a bleak mystery shaped "
                              'by grief and jurisdictional neglect.',
               'source_url': 'https://en.wikipedia.org/wiki/Wind_River_(film)',
               'note': 'Verified by tconst/title/year and reservation murder investigation plot; '
                       "old Shoshone chief's wife dream description belongs to another Wind "
                       'River.'},
 'tt0478087': {'description': 'An MIT math student joins a blackjack team that uses card-counting '
                              'schemes in Las Vegas, risking ambition, friendship, and his future '
                              'under casino scrutiny.',
               'source_url': 'https://en.wikipedia.org/wiki/21_(2008_film)',
               'note': 'Verified by tconst/title/year and MIT blackjack-team premise; old '
                       'final-fling-in-Vegas description belongs to another 21.'},
 'tt1601913': {'description': 'After a plane crash in the Alaskan wilderness, traumatized oil '
                              'workers follow a hardened survivor while wolves, cold, and despair '
                              'close in.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Grey_(film)',
               'note': 'Verified by tconst/title/year and Alaskan survival thriller; old Tehran '
                       'infidelity description belongs to another The Grey.'},
 'tt6823368': {'description': 'David Dunn, Kevin Crumb, and Elijah Price are confined by a '
                              'psychiatrist who questions their superhuman beliefs, setting up a '
                              'psychological showdown about identity and power.',
               'source_url': 'https://en.wikipedia.org/wiki/Glass_(2019_film)',
               'note': 'Verified by tconst/title/year and Unbreakable/Split crossover plot; old '
                       'island-art description belongs to another Glass.'},
 'tt0976051': {'description': "A German teenager's affair with an older woman becomes a lifelong "
                              'moral wound when he later discovers her role as a Nazi '
                              'concentration camp guard.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Reader_(2008_film)',
               'note': 'Verified by tconst/title/year and Hanna Schmitz postwar guilt plot; old '
                       'kidnapped-translator briefcase description belongs to another The Reader.'},
 'tt3079380': {'description': 'Desk-bound CIA analyst Susan Cooper goes undercover in the field to '
                              'stop a stolen nuclear device, proving herself amid chaotic '
                              'espionage and bruising comedy.',
               'source_url': 'https://en.wikipedia.org/wiki/Spy_(2015_film)',
               'note': 'Verified by tconst/title/year and Susan Cooper CIA comedy premise; old '
                       '1916 Russia description belongs to another Spy.'},
 'tt1832382': {'description': "An Iranian couple's separation collides with caregiving, class "
                              "tension, and legal accusations after the husband's father is "
                              "injured while under another woman's care.",
               'source_url': 'https://en.wikipedia.org/wiki/A_Separation',
               'note': 'Verified by tconst/title/year/original title Jodaeiye Nader az Simin; old '
                       'one-line parents-phone-trip description belongs to another A Separation.'},
 'tt1798684': {'description': 'After personal tragedy destroys his boxing career and family life, '
                              'champion Billy Hope fights through grief, anger, and discipline to '
                              'reclaim himself and his daughter.',
               'source_url': 'https://en.wikipedia.org/wiki/Southpaw_(film)',
               'note': 'Verified by tconst/title/year and Billy Hope boxing drama premise; old '
                       'Irish Traveller documentary description belongs to another Southpaw.'},
 'tt0163978': {'description': 'A young backpacker follows a secret map to an idyllic Thai island '
                              'commune, where paradise curdles into jealousy, secrecy, and '
                              'violence.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Beach_(film)',
               'note': 'Verified by tconst/title/year and Thai island backpacker plot; old '
                       "doctor's-wife journalist description belongs to another The Beach."},
 'tt1655442': {'description': 'As silent cinema gives way to talkies, a fading movie star and a '
                              'rising actress navigate pride, reinvention, and affection in '
                              'black-and-white Hollywood melodrama.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Artist_(film)',
               'note': 'Verified by tconst/title/year and silent-film/talkies premise; old nurse '
                       'stealing drawings description belongs to another The Artist.'},
 'tt3741834': {'description': 'After being separated from his family as a child in India and '
                              'adopted in Australia, Saroo uses memory and Google Earth to search '
                              'for his birth mother.',
               'source_url': 'https://en.wikipedia.org/wiki/Lion_(2016_film)',
               'note': 'Verified by tconst/title/year and Saroo Brierley biographical plot; old '
                       'coma/conspiracy description belongs to another Lion.'},
 'tt1051906': {'description': 'After escaping an abusive partner, Cecilia is stalked by an unseen '
                              'presence and must prove that his technological obsession has '
                              'survived his apparent death.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Invisible_Man_(2020_film)',
               'note': 'Verified by tconst/title/year and Cecilia abusive-partner/invisibility '
                       'premise; old scientist-military-secret description belongs to a different '
                       'Invisible Man adaptation.'},
 'tt0462499': {'description': 'John Rambo is pulled out of isolation in Thailand to rescue '
                              'missionaries captured by a brutal military regime in war-torn '
                              'Myanmar.',
               'source_url': 'https://en.wikipedia.org/wiki/Rambo_(2008_film)',
               'note': 'Verified by tconst/title/year/display title Rambo 4 and Myanmar rescue '
                       'plot; old car dealer/trained pig description belongs to another Rambo.'},
 'tt2005151': {'description': 'Two young Miami arms dealers exploit Pentagon contracting loopholes '
                              'and chase a massive Afghan ammunition deal that exposes greed, '
                              'fraud, and betrayal.',
               'source_url': 'https://en.wikipedia.org/wiki/War_Dogs_(2016_film)',
               'note': 'Verified by tconst/title/year and arms-dealer contract plot; old '
                       'military-training dog description belongs to another War Dogs.'},
 'tt0265208': {'description': 'A high school senior falls for the charismatic girl next door, then '
                              'discovers her adult-film past and gets pulled into a risky lesson '
                              'about desire, image, and growing up.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Girl_Next_Door_(2004_film)',
               'note': 'Verified by tconst/title/year and teen romantic comedy premise; old '
                       'trafficking survivor description belongs to another The Girl Next Door.'},
 'tt4555426': {'description': 'Newly appointed prime minister Winston Churchill faces political '
                              'pressure, military disaster, and the choice between negotiating '
                              'with Hitler or rallying Britain to fight.',
               'source_url': 'https://en.wikipedia.org/wiki/Darkest_Hour_(film)',
               'note': 'Verified by tconst/title/year and Churchill wartime leadership premise; '
                       'old abandoned-summer-camp killer description belongs to another Darkest '
                       'Hour.'},
 'tt2316411': {'description': 'A withdrawn history professor spots his exact double in a film and '
                              'becomes obsessed with finding him, slipping into a surreal '
                              'psychological mystery of identity and desire.',
               'source_url': 'https://en.wikipedia.org/wiki/Enemy_(2013_film)',
               'note': 'Verified by tconst/title/year and Denis Villeneuve doppelganger plot; old '
                       'childhood-rivals description belongs to another Enemy.'},
 'tt1606389': {'description': "After a car crash erases Paige's memory of her marriage, her "
                              'husband Leo tries to rebuild their relationship while she is drawn '
                              'back toward her old life.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Vow_(2012_film)',
               'note': 'Verified by tconst/title/year and post-accident memory-loss romance '
                       'premise; old priest/nun songwriter description belongs to another The '
                       'Vow.'},
 'tt1355644': {'description': 'A malfunction wakes Jim and Aurora decades early on a colony ship, '
                              'forcing them into a lonely romance and a desperate fight to save '
                              'thousands still asleep.',
               'source_url': 'https://en.wikipedia.org/wiki/Passengers_(2016_film)',
               'note': 'Verified by tconst/title/year and interstellar hibernation premise; old '
                       'one-night marriage description belongs to another Passengers.'},
 'tt0105695': {'description': 'A retired outlaw takes one last bounty job in a bleak Western where '
                              'revenge, aging, violence, and myth collide without heroism.',
               'source_url': 'https://en.wikipedia.org/wiki/Unforgiven',
               'note': 'Verified by tconst/title/year and William Munny western plot; old generic '
                       'betrayal description is not the 1992 Clint Eastwood film.'},
 'tt0353969': {'description': 'Two detectives in 1980s South Korea hunt a serial rapist and '
                              'murderer, but botched evidence, pressure, and obsession corrode the '
                              'investigation.',
               'source_url': 'https://en.wikipedia.org/wiki/Memories_of_Murder',
               'note': 'Verified by tconst/title/year/original title Salinui chueok; old '
                       'amnesiac-family-threat description belongs to the 1990 American television '
                       'film.'},
 'tt1872194': {'description': "A slick defense lawyer returns home for his mother's funeral and "
                              'must defend his estranged father, a respected judge accused of '
                              'murder.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Judge_(2014_film)',
               'note': 'Verified by tconst/title/year and father-son courtroom premise; old Aaron '
                       'Barak documentary description belongs to another The Judge.'},
 'tt3532216': {'description': 'Pilot Barry Seal is recruited by the CIA, pulled into drug '
                              'smuggling for the Medellin Cartel, and trapped in a frantic web of '
                              'money, politics, and betrayal.',
               'source_url': 'https://en.wikipedia.org/wiki/American_Made_(film)',
               'note': 'Verified by tconst/title/year and Barry Seal/Tom Cruise plot; old '
                       'Sikh-family desert-highway description belongs to another American Made.'},
 'tt3631112': {'description': 'A troubled commuter fixates on a seemingly perfect couple from her '
                              'train window, then becomes entangled in a missing-person mystery '
                              'that exposes buried trauma and deception.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Girl_on_the_Train_(2016_film)',
               'note': 'Verified by tconst/title/year and Rachel Watson thriller plot; old false '
                       'anti-Semitic attack description belongs to another Girl on the Train.'},
 'tt1527186': {'description': 'As a rogue planet approaches Earth, two sisters face depression, '
                              'dread, and family fracture during a wedding and the possible end of '
                              'the world.',
               'source_url': 'https://en.wikipedia.org/wiki/Melancholia_(2011_film)',
               'note': 'Verified by tconst/title/year and Lars von Trier rogue-planet premise; old '
                       'writer-family-loss line belongs to another Melancholia.'},
 'tt1355683': {'description': 'Boston gangster Whitey Bulger expands his criminal empire while '
                              'acting as an FBI informant, exposing corruption, loyalty, and '
                              'violence in South Boston.',
               'source_url': 'https://en.wikipedia.org/wiki/Black_Mass_(film)',
               'note': 'Verified by tconst/title/year and Whitey Bulger crime-biography plot; old '
                       'WWII supernatural-soldiers description belongs to another Black Mass.'},
 'tt2066051': {'description': 'Elton John recounts his rise from shy prodigy Reginald Dwight to '
                              'flamboyant rock star while confronting addiction, loneliness, '
                              'sexuality, and artistic identity.',
               'source_url': 'https://en.wikipedia.org/wiki/Rocketman_(film)',
               'note': 'Verified by tconst/title/year and Elton John musical biopic; old homemade '
                       'flat-Earth rocket documentary description belongs to another Rocketman.'},
 'tt0448694': {'description': 'Before meeting Shrek, sword-fighting outlaw Puss teams with Kitty '
                              'Softpaws and Humpty Dumpty in a fairy-tale heist involving magic '
                              'beans and old betrayal.',
               'source_url': 'https://en.wikipedia.org/wiki/Puss_in_Boots_(2011_film)',
               'note': 'Verified by tconst/title/year and DreamWorks Puss origin plot; old '
                       "miller's cat folktale summary belongs to a different Puss in Boots "
                       'adaptation.'},
 'tt1560747': {'description': 'A traumatized World War II veteran drifts into the orbit of a '
                              'charismatic movement leader, forming a volatile bond around faith, '
                              'control, and damaged masculinity.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Master_(2012_film)',
               'note': 'Verified by tconst/title/year and Freddie Quell/Lancaster Dodd premise; '
                       'old taekwondo-family description belongs to another The Master.'},
 'tt3470600': {'description': 'A desperate koala theater owner stages a singing competition, '
                              'drawing insecure performers whose dreams, families, and failures '
                              'converge on one chaotic show.',
               'source_url': 'https://en.wikipedia.org/wiki/Sing_(2016_American_film)',
               'note': 'Verified by tconst/title/year and Illumination singing-contest plot; old '
                       'Brooklyn high-school talent-show description belongs to another Sing.'},
 'tt5104604': {'description': 'In a dystopian Japanese city, a boy travels to Trash Island to find '
                              'his banished dog, joining a pack of exiled dogs against political '
                              'cruelty.',
               'source_url': 'https://en.wikipedia.org/wiki/Isle_of_Dogs_(film)',
               'note': 'Verified by tconst/title/year and Wes Anderson stop-motion dog-exile plot; '
                       'old London mob-boss affair description belongs to another Isle of Dogs.'},
 'tt12801262': {'description': 'On the Italian Riviera, young sea monster Luca experiences '
                               'friendship, freedom, and first steps toward identity while hiding '
                               'his true nature from humans.',
                'source_url': 'https://en.wikipedia.org/wiki/Luca_(2021_film)',
                'note': 'Verified by tconst/title/year and Pixar sea-monster coming-of-age '
                        'premise; old Chilean dictatorship recovery description belongs to another '
                        'Luca.'},
 'tt7798634': {'description': "A bride's wedding night becomes a deadly hide-and-seek ritual when "
                              'her wealthy in-laws hunt her through their mansion to satisfy a '
                              'family pact.',
               'source_url': 'https://en.wikipedia.org/wiki/Ready_or_Not_(2019_film)',
               'note': 'Verified by tconst/title/year and lethal in-law game premise; old Las '
                       'Vegas bachelor-party description belongs to another Ready or Not.'},
 'tt4178092': {'description': "A married couple's new life is disrupted by an old acquaintance "
                              'whose gifts and memories expose a buried act of cruelty from the '
                              "husband's past.",
               'source_url': 'https://en.wikipedia.org/wiki/The_Gift_(2015_American_film)',
               'note': 'Verified by tconst/title/year and Joel Edgerton psychological thriller '
                       "plot; old Asperger's romance description belongs to another The Gift."},
 'tt1980929': {'description': 'A burned-out music executive discovers a heartbroken songwriter in '
                              'New York, and together they record an outdoor album that helps both '
                              'rebuild their lives.',
               'source_url': 'https://en.wikipedia.org/wiki/Begin_Again_(film)',
               'note': 'Verified by tconst/title/year and New York songwriter/producer plot; old '
                       'foster-care description belongs to another Begin Again.'},
 'tt8079248': {'description': 'After a global blackout leaves everyone else forgetting the '
                              'Beatles, a struggling musician becomes famous by performing their '
                              'songs and wrestling with love, honesty, and success.',
               'source_url': 'https://en.wikipedia.org/wiki/Yesterday_(2019_film)',
               'note': 'Verified by tconst/title/year and Beatles-memory premise; old '
                       'college-romance line belongs to another Yesterday.'},
 'tt13833688': {'description': 'A reclusive, severely obese English teacher tries to reconnect '
                               'with his estranged teenage daughter while grief, shame, faith, and '
                               'mortality close in.',
                'source_url': 'https://en.wikipedia.org/wiki/The_Whale_(2022_film)',
                'note': 'Verified by tconst/title/year and Charlie estranged-daughter drama; old '
                        'orca description belongs to another The Whale.'},
 'tt6266538': {'description': "A darkly comic biographical drama traces Dick Cheney's rise to "
                              'power and the political machinery that reshaped American government '
                              'and war policy.',
               'source_url': 'https://en.wikipedia.org/wiki/Vice_(2018_film)',
               'note': 'Verified by tconst/title/year and Dick Cheney biopic premise; old '
                       'topless-club courtroom description belongs to another Vice.'},
 'tt15791034': {'description': 'A woman arrives at a Detroit rental house that has been '
                               'double-booked, then discovers a hidden basement and a horrifying '
                               'secret beneath the property.',
                'source_url': 'https://en.wikipedia.org/wiki/Barbarian_(2022_film)',
                'note': 'Verified by tconst/title/year and Detroit rental-house horror plot; old '
                        'mining-rights romance description belongs to another Barbarian.'},
 'tt1403981': {'description': 'A grieving college student falls for a young woman while both '
                              'confront family wounds, anger, and fragile connection in New York '
                              'City.',
               'source_url': 'https://en.wikipedia.org/wiki/Remember_Me_(2010_film)',
               'note': 'Verified by tconst/title/year and Tyler/Ally romantic drama premise; old '
                       'British ex-girlfriend visit description belongs to another Remember Me.'},
 'tt4172430': {'description': 'A security team at the U.S. diplomatic compound in Benghazi fights '
                              'through the 2012 militant attacks, balancing duty, chaos, and '
                              'survival under siege.',
               'source_url': 'https://en.wikipedia.org/wiki/13_Hours:_The_Secret_Soldiers_of_Benghazi',
               'note': 'Verified by tconst/title/year and Benghazi attack premise; old '
                       'blizzard-passenger description belongs to another 13 Hours.'},
 'tt1334260': {'description': 'In an alternate England, three friends raised at a boarding school '
                              'confront love, memory, and the grim truth that they were created as '
                              'organ donors.',
               'source_url': 'https://en.wikipedia.org/wiki/Never_Let_Me_Go_(2010_film)',
               'note': 'Verified by tconst/title/year and Kazuo Ishiguro clone-donor premise; old '
                       'reporter smuggling ballerina wife description belongs to another Never Let '
                       'Me Go.'},
 'tt5022702': {'description': 'A deaf writer living alone in the woods fights for survival when a '
                              'masked killer stalks her isolated home in a tense, stripped-down '
                              'thriller.',
               'source_url': 'https://en.wikipedia.org/wiki/Hush_(2016_feature_film)',
               'note': 'Verified by tconst/title/year and Mike Flanagan home-invasion premise; old '
                       'forensic-psychologist patient description belongs to another Hush.'},
 'tt0245844': {'description': 'Wrongfully imprisoned Edmond Dantes escapes, reinvents himself as '
                              'the Count of Monte Cristo, and seeks revenge on the friend who '
                              'betrayed him.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Count_of_Monte_Cristo_(2002_film)',
               'note': 'Verified by tconst/title/year and 2002 Kevin Reynolds adaptation; old Jean '
                       'Marais/Lia Amanda description belongs to an earlier adaptation.'},
 'tt2854926': {'description': 'A group of adult friends continue their decades-long game of tag, '
                              'launching one last elaborate chase before their undefeated friend '
                              'gets married.',
               'source_url': 'https://en.wikipedia.org/wiki/Tag_(2018_film)',
               'note': 'Verified by tconst/title/year and adult tag-game comedy premise; old '
                       'graffiti-artist description belongs to another Tag.'},
 'tt1588170': {'description': 'After his fiancee is murdered by a sadistic serial killer, a secret '
                              'agent begins a brutal revenge game that turns hunter and prey into '
                              'mirrors of violence.',
               'source_url': 'https://en.wikipedia.org/wiki/I_Saw_the_Devil',
               'note': 'Verified by tconst/title/year/original title Ang-ma-reul bo-at-da; old '
                       'asylum/therapist description belongs to another I Saw the Devil.'},
 'tt0765010': {'description': 'When a Marine captain presumed dead returns from captivity in '
                              'Afghanistan, trauma and suspicion fracture his marriage and his '
                              'bond with his troubled brother.',
               'source_url': 'https://en.wikipedia.org/wiki/Brothers_(2009_film)',
               'note': 'Verified by tconst/title/year and Sam/Tommy war-family plot; old young '
                       'Britons vacation description belongs to another Brothers.'},
 'tt3531824': {'description': 'A shy high-school senior joins an anonymous online dare game, where '
                              'escalating public challenges turn romance, status, and digital '
                              'spectatorship into danger.',
               'source_url': 'https://en.wikipedia.org/wiki/Nerve_(2016_film)',
               'note': 'Verified by tconst/title/year and online dare-game premise; old wife/lover '
                       'closure description belongs to another Nerve.'},
 'tt13560574': {'description': 'In 1979 Texas, a film crew shooting an adult movie at an isolated '
                               'farmhouse is stalked by elderly hosts whose desire and resentment '
                               'turn murderous.',
                'source_url': 'https://en.wikipedia.org/wiki/X_(2022_film)',
                'note': 'Verified by tconst/title/year and Ti West slasher premise; old voyeur '
                        'charity-ball description belongs to another X.'},
 'tt2305051': {'description': 'After grief and addiction leave her life in ruins, Cheryl Strayed '
                              'hikes the Pacific Crest Trail alone, seeking endurance, memory, and '
                              'a way back to herself.',
               'source_url': 'https://en.wikipedia.org/wiki/Wild_(2014_film)',
               'note': 'Verified by tconst/title/year and Cheryl Strayed biographical hiking plot; '
                       'old woman-after-wolf description belongs to another Wild.'},
 'tt8009428': {'description': 'A burned-out basketball scout discovers a raw Spanish prospect and '
                              'risks his career to prepare him for the NBA while both men chase '
                              'redemption.',
               'source_url': 'https://en.wikipedia.org/wiki/Hustle_(2022_film)',
               'note': 'Verified by tconst/title/year and Adam Sandler basketball scout plot; old '
                       'Pete Rose gambling description belongs to another Hustle.'},
 'tt15474916': {'description': "After witnessing a patient's terrifying suicide, a psychiatrist "
                               'becomes haunted by a contagious smiling entity that feeds on '
                               'trauma and disbelief.',
                'source_url': 'https://en.wikipedia.org/wiki/Smile_(2022_film)',
                'note': 'Verified by tconst/title/year and supernatural trauma-curse premise; old '
                        'sell-happiness description belongs to another Smile.'},
 'tt3488710': {'description': 'French high-wire artist Philippe Petit plans and performs an '
                              'illegal walk between the Twin Towers, driven by obsession, risk, '
                              'and theatrical wonder.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Walk_(2015_film)',
               'note': 'Verified by tconst/title/year and Philippe Petit World Trade Center walk '
                       'plot; old boy/older Jewish man walking description belongs to another The '
                       'Walk.'},
 'tt0870984': {'description': "After their child's death, a grieving couple retreats to a forest "
                              'cabin where therapy, guilt, nature, and violent psychosexual horror '
                              'unravel them.',
               'source_url': 'https://en.wikipedia.org/wiki/Antichrist_(film)',
               'note': 'Verified by tconst/title/year and Lars von Trier grief-cabin horror '
                       'premise; old possessed daughter description belongs to another '
                       'Antichrist.'},
 'tt0421239': {'description': 'On a red-eye flight to Miami, a hotel manager is coerced by a '
                              'charming stranger into aiding an assassination plot while her '
                              'father is threatened.',
               'source_url': 'https://en.wikipedia.org/wiki/Red_Eye_(2005_American_film)',
               'note': 'Verified by tconst/title/year and Wes Craven flight-assassination premise; '
                       'old Christmas Eve mother/daughter airport description belongs to another '
                       'Red Eye.'},
 'tt4481414': {'description': 'A single man raising his mathematically gifted niece fights his '
                              "mother for custody, trying to protect the child's ordinary "
                              'childhood and extraordinary talent.',
               'source_url': 'https://en.wikipedia.org/wiki/Gifted_(2017_film)',
               'note': 'Verified by tconst/title/year and Mary Adler custody/math plot; old '
                       'supernatural family alien-force description belongs to another Gifted.'},
 'tt10640346': {'description': 'In 1920s Hollywood, ambitious performers and fixers chase fame '
                               'through decadence, chaos, addiction, and the brutal transition '
                               'from silent films to sound.',
                'source_url': 'https://en.wikipedia.org/wiki/Babylon_(2022_film)',
                'note': 'Verified by tconst/title/year and Damien Chazelle 1920s Hollywood '
                        'premise; old U.S. release note is not a plot description.'},
 'tt13146488': {'description': 'After surviving The Suicide Squad, violent idealist Chris Smith '
                               'joins Project Butterfly and confronts alien parasites, family '
                               'trauma, and his warped idea of peace.',
                'source_url': 'https://en.wikipedia.org/wiki/Peacemaker_(TV_series)',
                'note': 'Verified by tconst/title/year/content type and DC Peacemaker premise; old '
                        'UN/Kurds negotiation description belongs to another Peacemaker.'},
 'tt1187064': {'description': 'A woman on a yacht trip enters an abandoned ocean liner and becomes '
                              'trapped in a looping nightmare of violence, doubles, and impossible '
                              'repetition.',
               'source_url': 'https://en.wikipedia.org/wiki/Triangle_(2009_British_film)',
               'note': 'Verified by tconst/title/year and British time-loop yacht thriller; old '
                       'art-exhibition bank-robbery description belongs to another Triangle.'},
 'tt8244784': {'description': 'A group of strangers wake in a field and discover they are being '
                              'hunted by wealthy elites, until one survivor turns the game back on '
                              'its organizers.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Hunt_(2020_film)',
               'note': 'Verified by tconst/title/year and 2020 satirical human-hunt premise; old '
                       'Spanish Civil War hunting description belongs to another The Hunt.'},
 'tt6394270': {'description': "Women at Fox News challenge Roger Ailes' abuse of power, exposing "
                              'the personal cost and institutional machinery behind a '
                              'sexual-harassment scandal.',
               'source_url': 'https://en.wikipedia.org/wiki/Bombshell_(2019_film)',
               'note': 'Verified by tconst/title/year and Fox News/Roger Ailes scandal premise; '
                       'old cancer-drug lab employee description belongs to another Bombshell.'},
 'tt1615160': {'description': 'A London restaurateur and former special forces soldier hunts the '
                              'terrorists who killed his daughter, pressuring a compromised Irish '
                              'politician for answers.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Foreigner_(2017_film)',
               'note': 'Verified by tconst/title/year and Jackie Chan revenge-thriller premise; '
                       'old computer-game tournament description belongs to another Foreigner.'},
 'tt0490215': {'description': 'Two Portuguese Jesuit priests travel to 17th-century Japan to find '
                              'their mentor and face persecution, doubt, and the terrible cost of '
                              'faith.',
               'source_url': 'https://en.wikipedia.org/wiki/Silence_(2016_film)',
               'note': 'Verified by tconst/title/year and Scorsese Jesuit mission plot; old '
                       'autistic boy/hermit description belongs to another Silence.'},
 'tt16419074': {'description': "Nike executive Sonny Vaccaro risks the company's basketball budget "
                               'to sign rookie Michael Jordan, creating the Air Jordan deal that '
                               'reshapes sports marketing.',
                'source_url': 'https://en.wikipedia.org/wiki/Air_(2023_American_film)',
                'note': 'Verified by tconst/title/year and Air Jordan business origin plot; old '
                        'winged-creatures anime description belongs to another Air.'},
 'tt2139881': {'description': 'A scrappy journalist reconnects with his former babysitter, now a '
                              'presidential candidate, and their unlikely romance collides with '
                              'campaign optics and political pressure.',
               'source_url': 'https://en.wikipedia.org/wiki/Long_Shot_(2019_film)',
               'note': 'Verified by tconst/title/year and Seth Rogen/Charlize Theron '
                       'political-romance plot; old alibi-footage description belongs to another '
                       'Long Shot.'},
 'tt0460791': {'description': 'In a 1920s hospital, an injured stuntman tells a young girl a '
                              'fantastical adventure story that reflects his despair and her '
                              'imagination.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Fall_(2006_film)',
               'note': 'Verified by tconst/title/year and Tarsem Singh hospital fantasy premise; '
                       'old 1960s violence/revolution statement belongs to another The Fall.'},
 'tt1656190': {'description': 'A disgraced fighter protects a gifted young girl whose memorized '
                              'code makes her a target for Triads, Russian mobsters, and corrupt '
                              'police.',
               'source_url': 'https://en.wikipedia.org/wiki/Safe_(2012_film)',
               'note': 'Verified by tconst/title/year and Jason Statham action-thriller premise; '
                       'old environmental illness description belongs to another Safe.'},
 'tt1450321': {'description': 'A corrupt Edinburgh detective chasing promotion spirals through '
                              'drugs, manipulation, hallucination, and self-destruction while '
                              'investigating a murder.',
               'source_url': 'https://en.wikipedia.org/wiki/Filth_(film)',
               'note': 'Verified by tconst/title/year and Irvine Welsh/Bruce Robertson plot; old '
                       'Nigerian psychologist description belongs to another Filth.'},
 'tt0340163': {'description': 'A traumatized police chief is pulled into a mansion hostage crisis '
                              'while criminals also threaten his own family to recover '
                              'incriminating evidence.',
               'source_url': 'https://en.wikipedia.org/wiki/Hostage_(2005_film)',
               'note': 'Verified by tconst/title/year and Bruce Willis hostage-negotiator premise; '
                       'old hijacked-flight rescue description belongs to another Hostage.'},
 'tt1063669': {'description': "A German teacher's classroom experiment in authoritarian discipline "
                              'grows into a dangerous movement, exposing how quickly group '
                              'identity can become fascistic.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Wave_(2008_film)',
               'note': 'Verified by tconst/title/year/original title Die Welle; old imperfect '
                       'romance description belongs to another The Wave.'},
 'tt1682180': {'description': "After her father's death, isolated India Stoker is drawn to her "
                              'mysterious uncle, uncovering family secrets, desire, and a violent '
                              'inheritance.',
               'source_url': 'https://en.wikipedia.org/wiki/Stoker_(film)',
               'note': 'Verified by tconst/title/year and Park Chan-wook family thriller premise; '
                       'old betrayed husband plantation description belongs to another Stoker.'},
 'tt0848537': {'description': 'A teenage girl is magically shrunk into a hidden forest world where '
                              'tiny warriors battle decay and she must help protect the life of '
                              'nature.',
               'source_url': 'https://en.wikipedia.org/wiki/Epic_(2013_film)',
               'note': 'Verified by tconst/title/year and Blue Sky animated forest-war premise; '
                       'old wild-dogs dinosaur description belongs to another Epic.'},
 'tt2224026': {'description': 'After an alien race relocates humanity, a misfit Boov named Oh '
                              'teams with a girl searching for her mother in a colorful story '
                              'about belonging and trust.',
               'source_url': 'https://en.wikipedia.org/wiki/Home_(2015_film)',
               'note': 'Verified by tconst/title/year and DreamWorks Boov/Tip premise; old '
                       'homeless teenagers description belongs to another Home.'},
 'tt1570728': {'description': 'A newly separated father learns dating confidence from a slick '
                              'bachelor while tangled romances around his family expose pain, '
                              'longing, and the messy hope of second chances.',
               'source_url': 'https://en.wikipedia.org/wiki/Crazy%2C_Stupid%2C_Love',
               'note': 'Verified by tconst/title/year against Wikipedia: 2011 romantic comedy '
                       'about Cal Weaver, Jacob Palmer, and interconnected love stories, not a '
                       'long-distance relationship story.'},
 'tt1270797': {'description': 'A disgraced journalist bonds with an alien symbiote after exposing '
                              'a biotech corporation, becoming a violent antihero caught between '
                              'human vulnerability, body horror, and chaotic comic-book action.',
               'source_url': 'https://en.wikipedia.org/wiki/Venom_(2018_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2018 film about Eddie '
                       'Brock, the Life Foundation, and the Venom symbiote; old description '
                       'described unrelated scenarios.'},
 'tt3281548': {'description': 'Greta Gerwig reframes the March sisters coming of age in Civil '
                              "War-era Massachusetts, centering Jo's artistic ambition, family "
                              'devotion, romance, grief, and the compromises women face.',
               'source_url': 'https://en.wikipedia.org/wiki/Little_Women_(2019_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 Greta Gerwig period '
                       'drama adaptation, not a musical adaptation.'},
 'tt0342258': {'description': 'Raised as a weapon by a Glasgow crime boss, Danny begins to reclaim '
                              'gentleness and personhood after a blind piano tuner and his '
                              'stepdaughter offer him shelter from violence.',
               'source_url': 'https://en.wikipedia.org/wiki/Unleashed_(2005_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2005 Jet Li action drama '
                       'about Danny, Bart, and Sam; old Transylvania hunting-party description was '
                       'unrelated.'},
 'tt1172049': {'description': 'After his wife dies in a car crash, an emotionally numb investment '
                              'banker forms an unlikely bond with a customer-service worker and '
                              'starts dismantling his life to understand his grief.',
               'source_url': 'https://en.wikipedia.org/wiki/Demolition_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 Jake Gyllenhaal '
                       'comedy-drama about Davis Mitchell grieving his wife; old courier-package '
                       'plot was unrelated.'},
 'tt0404390': {'description': 'A low-level mobster races through a violent underworld to recover a '
                              'gun used in a corrupt-cop killing after a neighborhood boy steals '
                              'it from his basement.',
               'source_url': 'https://en.wikipedia.org/wiki/Running_Scared_(2006_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2006 Paul Walker neo-noir '
                       'thriller about Joey Gazelle and a missing gun; old Cuba/Everglades '
                       'description was unrelated.'},
 'tt0453562': {'description': "Jackie Robinson breaks Major League Baseball's color barrier with "
                              'the Brooklyn Dodgers, facing racist hostility while Branch Rickey '
                              'backs his talent, discipline, and historic courage.',
               'source_url': 'https://en.wikipedia.org/wiki/42_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2013 Jackie Robinson '
                       'biopic starring Chadwick Boseman; old Quit India Movement description was '
                       'unrelated.'},
 'tt0963178': {'description': 'An Interpol agent and a New York prosecutor investigate a powerful '
                              'global bank involved in arms deals, political corruption, and '
                              'murder, pushing a conspiracy thriller across borders.',
               'source_url': 'https://en.wikipedia.org/wiki/The_International_(2009_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2009 thriller starring '
                       'Clive Owen and Naomi Watts about a corrupt international bank; old '
                       'street-performer protest description was unrelated.'},
 'tt9071322': {'description': 'Corporate lawyer Robert Bilott risks his career and family '
                              "stability to expose DuPont's toxic chemical contamination, turning "
                              "a farmer's plea into a years-long legal fight.",
               'source_url': 'https://en.wikipedia.org/wiki/Dark_Waters_(2019_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 legal thriller about '
                       'Bilott, DuPont, and contaminated West Virginia farmland; old '
                       'monastery/island description was unrelated.'},
 'tt1781922': {'description': 'An American engineer and his family are trapped in a Southeast '
                              'Asian coup, forcing them through a tense survival escape amid '
                              'riots, executions, and collapsing order.',
               'source_url': 'https://en.wikipedia.org/wiki/No_Escape_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 Owen Wilson action '
                       'thriller about an expat family caught in a coup; old prison-island '
                       'description was unrelated.'},
 'tt0402399': {'description': 'Terrence Malick meditates on the Jamestown settlement through '
                              'Pocahontas, John Smith, and John Rolfe, blending colonial '
                              'encounter, romance, cultural rupture, and lyrical historical drama.',
               'source_url': 'https://en.wikipedia.org/wiki/The_New_World_(2005_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2005 Terrence Malick '
                       'historical drama about Jamestown and Pocahontas; old lesbian-couple family '
                       'description was unrelated.'},
 'tt3850214': {'description': 'A 1990s-obsessed Inglewood teen and his friends are pulled into a '
                              'dangerous drug deal, turning a coming-of-age comedy into a sharp '
                              'story about identity and survival.',
               'source_url': 'https://en.wikipedia.org/wiki/Dope_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 coming-of-age '
                       'comedy-drama about Malcolm in Inglewood; old London drug-culture '
                       'description was unrelated.'},
 'tt9170108': {'description': 'Two androids attempt to raise human children on a remote planet '
                              "after Earth's destruction, as faith, programming, survival, and "
                              'violent human factions threaten their mission.',
               'source_url': 'https://en.wikipedia.org/wiki/Raised_by_Wolves_(American_TV_series)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2020 HBO Max '
                       'science-fiction series about android parents on Kepler-22b; old Caitlin '
                       'Moran childhood description was unrelated.'},
 'tt1619029': {'description': "In the days after John F. Kennedy's assassination, Jacqueline "
                              'Kennedy battles shock and grief while shaping the public memory of '
                              "her husband's presidency and legacy.",
               'source_url': 'https://en.wikipedia.org/wiki/Jackie_(2016_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2016 Pablo Larrain film '
                       'starring Natalie Portman as Jacqueline Kennedy; old father-kidnapping '
                       'description was unrelated.'},
 'tt12536294': {'description': 'During a tense royal Christmas at Sandringham, Princess Diana '
                               'confronts isolation, surveillance, motherhood, and psychological '
                               'strain as she considers leaving her marriage and public role.',
                'source_url': 'https://en.wikipedia.org/wiki/Spencer_(film)',
                'note': 'Verified by tconst/title/year against Wikipedia: 2021 Pablo Larrain '
                        'psychological drama about Diana, Princess of Wales; old mail-clerk '
                        'advertising-firm description was unrelated.'},
 'tt0995039': {'description': 'A misanthropic New York dentist briefly dies and wakes able to see '
                              'ghosts, then gets drawn into a funny, melancholy romance while '
                              'helping the dead settle unfinished business.',
               'source_url': 'https://en.wikipedia.org/wiki/Ghost_Town_(2008_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2008 Ricky Gervais '
                       'fantasy comedy about a dentist who sees ghosts; old remote-China village '
                       'description was unrelated.'},
 'tt0861689': {'description': 'When a mysterious epidemic blinds nearly everyone in a city, one '
                              'sighted woman hides her immunity and guides a quarantined group '
                              'through fear, brutality, and social collapse.',
               'source_url': 'https://en.wikipedia.org/wiki/Blindness_(2008_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2008 thriller about a '
                       'mass blindness epidemic; old gun-bearing convict description was '
                       'unrelated.'},
 'tt0219699': {'description': 'A widowed clairvoyant in rural Georgia is pulled into a murder '
                              'investigation, where visions, abuse, suspicion, and small-town '
                              'secrets create a Southern Gothic supernatural mystery.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Gift_(2000_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2000 Sam Raimi '
                       'supernatural thriller about psychic Annie Wilson; old Asperger romance '
                       'description was unrelated.'},
 'tt0391304': {'description': 'Across seven years of chance meetings, Oliver and Emily drift from '
                              'attraction to friendship to complicated romance, testing timing, '
                              'ambition, fear, and emotional readiness.',
               'source_url': 'https://en.wikipedia.org/wiki/A_Lot_like_Love',
               'note': 'Verified by tconst/title/year against Wikipedia: 2005 romantic '
                       'comedy-drama about Oliver and Emily over seven years; old heiress/husband '
                       'weekend description was unrelated.'},
 'tt0250258': {'description': 'Volunteers in a simulated prison experiment are assigned prisoner '
                              'and guard roles, but the study spirals into humiliation, '
                              'authoritarian abuse, panic, and psychological violence.',
               'source_url': 'https://en.wikipedia.org/wiki/Das_Experiment',
               'note': 'Verified by tconst/title/year against Wikipedia: 2001 German thriller '
                       'based on a Stanford-prison-like experiment; old Hurricane Katrina '
                       'school-system description was unrelated.'},
 'tt0986233': {'description': 'Steve McQueen depicts the 1981 Irish hunger strike through Bobby '
                              'Sands and Maze Prison inmates, using austere imagery to explore '
                              'protest, bodily suffering, and political conviction.',
               'source_url': 'https://en.wikipedia.org/wiki/Hunger_(2008_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2008 historical drama '
                       'about the Irish hunger strike; old desperate-family food description was '
                       'unrelated.'},
 'tt3322364': {'description': 'Forensic pathologist Bennet Omalu discovers football-related brain '
                              'trauma in former NFL players and faces institutional resistance as '
                              'he pushes the league to confront CTE.',
               'source_url': 'https://en.wikipedia.org/wiki/Concussion_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 Will Smith '
                       'biographical sports drama about Bennet Omalu and CTE; old description '
                       'belonged to the unrelated 2013 film Concussion.'},
 'tt0406816': {'description': 'A legendary Coast Guard rescue swimmer, haunted by a fatal crash, '
                              'becomes an instructor and trains a talented but reckless recruit in '
                              'sacrifice, courage, and survival at sea.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Guardian_(2006_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2006 Kevin Costner/Ashton '
                       'Kutcher Coast Guard rescue-swimmer drama; old young-girl/grief description '
                       'was unrelated.'},
 'tt2140379': {'description': 'A dying billionaire transfers his consciousness into a younger '
                              'body, then uncovers the human cost of the procedure and is hunted '
                              'through a paranoid sci-fi conspiracy.',
               'source_url': 'https://en.wikipedia.org/wiki/Self/less',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 Tarsem Singh film '
                       'about Damian Hale and consciousness transfer; old identity-thief architect '
                       'description was unrelated.'},
 'tt8623904': {'description': 'A cynical London Christmas-shop worker recovering from serious '
                              'illness forms a mysterious bond with Tom, nudging her toward family '
                              'reconciliation, generosity, and emotional renewal.',
               'source_url': 'https://en.wikipedia.org/wiki/Last_Christmas_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 Paul Feig romantic '
                       'comedy about Kate and Tom; old angel/father description was inaccurate for '
                       'this title.'},
 'tt0439572': {'description': 'Barry Allen travels back in time to save his mother, fractures the '
                              'timeline, and joins alternate versions of himself and Batman in a '
                              'multiverse superhero crisis.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Flash_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2023 DC superhero film '
                       'about Barry Allen, time travel, and the multiverse; old police-chief '
                       'daughter/gambling-house description was unrelated.'},
 'tt1433811': {'description': 'Interlinked stories follow families, reporters, teens, and '
                              'identity-theft victims damaged by online intimacy, exploitation, '
                              'bullying, and isolation in a tense digital-age drama.',
               'source_url': 'https://en.wikipedia.org/wiki/Disconnect_(2012_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2012 ensemble drama about '
                       'negative effects of modern communication technology; old magical '
                       'cell-phone description was unrelated.'},
 'tt1527788': {'description': 'A withdrawn pawnshop owner with a violent past launches a ruthless '
                              'rescue mission after drug traffickers kidnap the young neighbor who '
                              'has become his only friend.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Man_from_Nowhere_(2010_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2010 South Korean action '
                       'thriller about Cha Tae-sik and kidnapped So-mi; old western saloon '
                       'description was unrelated.'},
 'tt1226753': {'description': 'Former Mossad agents confront the truth behind a celebrated mission '
                              'to capture a Nazi war criminal, as past lies and present guilt '
                              'collide in a tense espionage thriller.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Debt_(2010_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2010 thriller about '
                       'Mossad agents, Rachel Singer, and Dieter Vogel; old Argentine '
                       'schoolteacher description was unrelated.'},
 'tt3716530': {'description': 'A powerful video-game executive responds to a brutal assault with '
                              'unsettling control, drawing herself into a provocative '
                              'psychological game of trauma, power, desire, and revenge.',
               'source_url': 'https://en.wikipedia.org/wiki/Elle_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2016 Paul Verhoeven '
                       'psychological thriller starring Isabelle Huppert as Michele Leblanc; old '
                       'car-crash romance description was unrelated.'},
 'tt6850820': {'description': 'After cartel gunmen murder her husband and daughter and the justice '
                              'system fails her, Riley North disappears and returns as a '
                              'relentless vigilante seeking revenge.',
               'source_url': 'https://en.wikipedia.org/wiki/Peppermint_(2018_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2018 Jennifer Garner '
                       'vigilante thriller; old aircraft-engineer reminiscence description was '
                       'unrelated.'},
 'tt8637428': {'description': 'A Chinese American writer returns to China for a staged wedding so '
                              'the family can quietly gather around her terminally ill grandmother '
                              'without revealing the diagnosis.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Farewell_(2019_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 Lulu Wang film about '
                       'Billi, Nai Nai, and a concealed illness; old soccer-star description was '
                       'unrelated.'},
 'tt0472160': {'description': 'A sheltered heiress born with a pig-like nose because of a family '
                              'curse leaves home, seeking self-acceptance and love beyond '
                              'aristocratic expectations.',
               'source_url': 'https://en.wikipedia.org/wiki/Penelope_(2006_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2006 fantasy romantic '
                       'comedy about Penelope Wilhern and a family curse; old castle-tax '
                       'description was unrelated.'},
 'tt4244998': {'description': 'In prehistoric Europe, a young hunter left for dead after a failed '
                              'bison hunt befriends an injured wolf, and together they struggle '
                              'through the wilderness toward home.',
               'source_url': 'https://en.wikipedia.org/wiki/Alpha_(2018_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2018 prehistoric '
                       'adventure about Keda and a wolf; old punishment-under-hanged-corpse '
                       'description was unrelated.'},
 'tt2121382': {'description': 'A Swedish family vacationing in the Alps is shaken after the father '
                              'flees an apparent avalanche, exposing fear, resentment, and cracks '
                              'in the marriage.',
               'source_url': 'https://en.wikipedia.org/wiki/Force_Majeure_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2014 Ruben Ostlund black '
                       'comedy about an avalanche and marital tension; old Orient/drug-trafficking '
                       'description was unrelated.'},
 'tt1704573': {'description': 'A beloved Texas funeral director forms a controlling friendship '
                              'with a wealthy widow, then murders her, in a darkly comic '
                              'true-crime portrait of small-town denial.',
               'source_url': 'https://en.wikipedia.org/wiki/Bernie_(2011_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2011 Richard Linklater '
                       'black comedy about Bernie Tiede and Marjorie Nugent; old '
                       'orphanage/trash-bin description was unrelated.'},
 'tt5610554': {'description': 'An exhausted mother of three accepts help from a night nanny, '
                              'forming an intimate bond that exposes burnout, identity loss, and '
                              'the fragile edges of postpartum life.',
               'source_url': 'https://en.wikipedia.org/wiki/Tully_(2018_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2018 Jason Reitman/Diablo '
                       'Cody film about Marlo and night nanny Tully; old womanizer/brother '
                       'description was unrelated.'},
 'tt4226388': {'description': 'A lonely Spanish woman in Berlin is swept into a dangerous dawn '
                              'with four men, as flirtation turns into a desperate one-take crime '
                              'thriller.',
               'source_url': 'https://en.wikipedia.org/wiki/Victoria_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 German one-take '
                       'thriller about Victoria, Sonne, and a Berlin crime; old Mojave settlement '
                       'description was unrelated.'},
 'tt3758172': {'description': 'Adventurer Yossi Ghinsberg follows a guide into the Bolivian Amazon '
                              'and becomes stranded, fighting hunger, injury, fear, and the jungle '
                              'to survive.',
               'source_url': 'https://en.wikipedia.org/wiki/Jungle_(2017_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2017 survival drama based '
                       'on Yossi Ghinsberg in the Amazon; old Indian princess/mammoth description '
                       'was unrelated.'},
 'tt6079516': {'description': 'A detective investigating a child abduction faces eerie '
                              'disturbances inside his own fractured home, where hidden intruders '
                              'and old crimes twist the mystery into horror.',
               'source_url': 'https://en.wikipedia.org/wiki/I_See_You_(2019_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 crime horror '
                       'thriller about Detective Greg Harper and a missing child case; old Dr. '
                       'Dutt spirit description was unrelated.'},
 'tt10272386': {'description': 'An elderly man experiencing dementia resists help from his '
                               'daughter as faces, rooms, and memories shift around him, creating '
                               'an intimate psychological portrait of decline.',
                'source_url': 'https://en.wikipedia.org/wiki/The_Father_(2020_film)',
                'note': 'Verified by tconst/title/year against Wikipedia: 2020 Florian Zeller '
                        'psychological drama starring Anthony Hopkins as a man with dementia; old '
                        'description wrongly framed the story as greedy children trying to '
                        'institutionalize him.'},
 'tt1245112': {'description': 'Immediately after the first outbreak, a tactical team enters the '
                              'quarantined Barcelona apartment building with a supposed medical '
                              'expert, uncovering religious and viral horrors inside.',
               'source_url': 'https://en.wikipedia.org/wiki/Rec_2',
               'note': 'Verified by tconst/title/year against Wikipedia: 2009 Spanish '
                       'found-footage sequel following soldiers and a scientist after the first '
                       'Rec; old description repeated the first film premise.'},
 'tt0116231': {'description': 'After 35 years in prison, aging bandit Baran travels to Istanbul '
                              'seeking the friend who betrayed him and the woman he lost, entering '
                              'a changed criminal world.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Bandit_(1996_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 1996 Turkish film about '
                       'Baran after a 35-year sentence; old World War II POW description was '
                       'unrelated.'},
 'tt6265828': {'description': 'After dying, a man returns as a sheet-covered ghost and silently '
                              'haunts the home he shared with his wife, drifting through grief, '
                              'memory, time, and attachment.',
               'source_url': 'https://en.wikipedia.org/wiki/A_Ghost_Story',
               'note': 'Verified by tconst/title/year against Wikipedia: 2017 David Lowery '
                       'supernatural drama about a dead man remaining in his house as a ghost; old '
                       'description was an overly generic romance and missed the premise.'},
 'tt7374948': {'description': 'Childhood friends Sasha and Marcus reconnect in San Francisco after '
                              'years apart, testing old chemistry against fame, insecurity, family '
                              'history, and sharply comic romantic timing.',
               'source_url': 'https://en.wikipedia.org/wiki/Always_Be_My_Maybe',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 romantic comedy '
                       'about Sasha Tran and Marcus Kim reconnecting; old '
                       'brokenhearted-love-advice description was not the film premise.'},
 'tt3312830': {'description': 'At a luxury Swiss Alps resort, retired composer Fred Ballinger and '
                              'filmmaker Mick Boyle reflect on aging, memory, art, desire, regret, '
                              'and the fragile pull of the past.',
               'source_url': 'https://en.wikipedia.org/wiki/Youth_(2015_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 Paolo Sorrentino '
                       'comedy-drama about Fred Ballinger and Mick Boyle at a Swiss Alps resort; '
                       'old Djibouti student-choice description was unrelated.'},
 'tt1232829': {'description': 'Two immature rookie cops go undercover as high school students to '
                              'stop a new synthetic drug ring, turning a buddy-cop assignment into '
                              'chaotic action comedy and arrested adolescence.',
               'source_url': 'https://en.wikipedia.org/wiki/21_Jump_Street_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2012 film follows Schmidt '
                       'and Jenko infiltrating a high school drug ring; old description '
                       'over-centered the captain and was not a useful plot summary.'},
 'tt0101414': {'description': "Bookish Belle takes her father's place as prisoner in an enchanted "
                              'castle, where a cursed prince must learn compassion and love before '
                              'time runs out.',
               'source_url': 'https://en.wikipedia.org/wiki/Beauty_and_the_Beast_(1991_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 1991 Disney animated film '
                       'about Belle, the Beast, Gaston, and an enchanted curse; old description '
                       'was generic and non-informative.'},
 'tt2771200': {'description': 'Belle enters an enchanted castle to save her father and gradually '
                              "challenges the Beast's isolation, while Gaston's vanity and fear "
                              'threaten the chance to break the curse.',
               'source_url': 'https://en.wikipedia.org/wiki/Beauty_and_the_Beast_(2017_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2017 live-action musical '
                       'remake starring Emma Watson and Dan Stevens; old description was generic '
                       'and non-informative.'},
 'tt0955308': {'description': "After King Richard's death, archer Robin Longstride returns to "
                              "England, assumes a fallen knight's identity, and becomes drawn into "
                              'royal politics, invasion threats, and rebellion.',
               'source_url': 'https://en.wikipedia.org/wiki/Robin_Hood_(2010_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2010 Ridley Scott film '
                       'with Robin Longstride as a crusader archer caught in succession and French '
                       'invasion conflict; old description was an oversimplified legend summary.'},
 'tt1067106': {'description': 'Miserly Ebenezer Scrooge is forced through a ghostly Christmas Eve '
                              'reckoning with his past, present, and possible future in Robert '
                              "Zemeckis' dark motion-capture adaptation.",
               'source_url': 'https://en.wikipedia.org/wiki/A_Christmas_Carol_(2009_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2009 Robert Zemeckis '
                       'animated adaptation starring Jim Carrey as Scrooge and the spirits; old '
                       'description was too generic for recommendations.'},
 'tt2671706': {'description': 'In 1950s Pittsburgh, former Negro leagues player Troy Maxson '
                              'struggles with racial disappointment, pride, marriage, fatherhood, '
                              'and control as his family absorbs the cost of his bitterness.',
               'source_url': 'https://en.wikipedia.org/wiki/Fences_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2016 Denzel Washington '
                       "adaptation of August Wilson's play about Troy, Rose, Cory, and family "
                       'conflict; old abstract partition description was not the film plot.'},
 'tt2788432': {'description': 'A true-crime anthology dramatizes notorious American legal '
                              'scandals, with self-contained seasons exploring celebrity, race, '
                              'media spectacle, power, and institutional failure.',
               'source_url': 'https://en.wikipedia.org/wiki/American_Crime_Story',
               'note': 'Verified by tconst/title/year against Wikipedia: 2016 FX anthology series '
                       'with independent seasons based on real criminal cases; old description was '
                       'too generic to be useful.'},
 'tt2358891': {'description': 'Jep Gambardella, an aging Roman writer and socialite, wanders '
                              'through lavish parties, memories, spiritual emptiness, and artistic '
                              'paralysis while searching for meaning beneath beauty.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Great_Beauty',
               'note': 'Verified by tconst/title/year against Wikipedia: 2013 Paolo Sorrentino '
                       'film about Jep Gambardella in Rome; old description reduced the premise to '
                       'a vague recollection of youth.'},
 'tt5618256': {'description': 'FBI profiler Jim Fitzgerald uses forensic linguistics to help '
                              'identify the Unabomber, turning a long bombing investigation into a '
                              'tense procedural about language, obsession, and pursuit.',
               'source_url': 'https://en.wikipedia.org/wiki/Manhunt_(2017_TV_series)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2017 first season '
                       'Manhunt: Unabomber follows the FBI hunt for Ted Kaczynski; old description '
                       'was generic and non-specific.'},
 'tt0213338': {'description': 'In 2071, bounty hunters aboard the spaceship Bebop chase criminals '
                              'across the solar system while their pasts, loneliness, jazz-soaked '
                              'style, and noir melancholy catch up with them.',
               'source_url': 'https://en.wikipedia.org/wiki/Cowboy_Bebop',
               'note': 'Verified by tconst/title/year against Wikipedia: 1998 space western anime '
                       'about the Bebop bounty-hunter crew; old description was technically '
                       'related but far too generic.'},
 'tt0096657': {'description': "Rowan Atkinson's nearly wordless comic loner turns ordinary "
                              'errands, holidays, and social rituals into absurd disasters through '
                              'childish selfishness, physical comedy, and oblivious improvisation.',
               'source_url': 'https://en.wikipedia.org/wiki/Mr._Bean',
               'note': 'Verified by tconst/title/year against Wikipedia: 1990 British sitcom '
                       'starring Rowan Atkinson as Mr. Bean; old description was generic and weak '
                       'for recommendations.'},
 'tt8134470': {'description': "A wealthy Manhattan therapist's polished life collapses after a "
                              'violent murder implicates her husband, exposing privilege, denial, '
                              'secrets, and psychological suspense.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Undoing',
               'note': 'Verified by tconst/title/year against Wikipedia: 2020 HBO miniseries '
                       'starring Nicole Kidman and Hugh Grant around a murder investigation; old '
                       'widespread-disaster wording was misleading.'},
 'tt10160804': {'description': 'Clint Barton tries to get home for Christmas while reluctantly '
                               'teaming with young archer Kate Bishop against enemies tied to his '
                               'violent past as Ronin.',
                'source_url': 'https://en.wikipedia.org/wiki/Hawkeye_(miniseries)',
                'note': 'Verified by tconst/title/year against Wikipedia: 2021 MCU miniseries '
                        'pairs Clint Barton with Kate Bishop after Avengers: Endgame; old '
                        'description was flippant and incomplete.'},
 'tt10189514': {'description': 'Inspired by low-cost aviation pioneer G. R. Gopinath, Nedumaaran '
                               'Rajangam battles class barriers, industry resistance, and personal '
                               'sacrifice to make air travel affordable.',
                'source_url': 'https://en.wikipedia.org/wiki/Soorarai_Pottru',
                'note': 'Verified by tconst/title/year against Wikipedia: 2020 Tamil drama '
                        'inspired by G. R. Gopinath and low-cost airline ambitions; old '
                        'description was vague but related.'},
 'tt3846674': {'description': "Shy Lara Jean's private love letters are secretly mailed to her "
                              'crushes, forcing her into a fake relationship that becomes a warm '
                              'teen romance about vulnerability and grief.',
               'source_url': 'https://en.wikipedia.org/wiki/To_All_the_Boys_I%27ve_Loved_Before_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2018 Netflix teen romance '
                       'about Lara Jean, mailed letters, and a fake relationship with Peter; old '
                       'description lacked the core fake-dating premise.'},
 'tt1258197': {'description': 'Eight job candidates are locked in a room for a mysterious final '
                              'exam with strict rules, turning collaboration into paranoia, '
                              'strategy, and psychological survival.',
               'source_url': 'https://en.wikipedia.org/wiki/Exam_(2009_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2009 thriller about '
                       'candidates facing a blank exam and shifting alliances; old description '
                       'oversimplified the premise.'},
 'tt2582802': {'description': 'An obsessive young jazz drummer at a prestigious conservatory is '
                              'pushed into psychological and physical extremes by an abusive '
                              'instructor who equates greatness with suffering.',
               'source_url': 'https://en.wikipedia.org/wiki/Whiplash_(2014_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2014 Damien Chazelle film '
                       'about Andrew Neiman and Terence Fletcher; old description was accurate but '
                       'too thin for mood recommendations.'},
 'tt8633478': {'description': 'A wheelchair-using homeschooled teen begins to suspect her '
                              'controlling mother is hiding the truth about her illness, turning '
                              'domestic care into claustrophobic psychological horror.',
               'source_url': 'https://en.wikipedia.org/wiki/Run_(2020_American_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2020 Aneesh Chaganty '
                       'thriller about Chloe and Diane; old description was accurate but omitted '
                       'the coercive mother-daughter conflict.'},
 'tt8291224': {'description': 'Major Vihaan Shergill leads a covert Indian military retaliation '
                              'after the 2016 Uri attack, blending patriotic war action with '
                              'grief, strategy, and special-forces resolve.',
               'source_url': 'https://en.wikipedia.org/wiki/Uri:_The_Surgical_Strike',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 Hindi war action '
                       'film about retaliation to the Uri attack; old description was accurate but '
                       'overly bare.'},
 'tt7826376': {'description': 'In a near future where consciousness can be uploaded to corporate '
                              'digital afterlives, programmer Nathan Brown navigates virtual '
                              'luxury, dependency, romance, and suspicions he was murdered.',
               'source_url': 'https://en.wikipedia.org/wiki/Upload_(TV_series)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2020 Greg Daniels series '
                       'about Nathan at Lakeview and Nora investigating his death; old description '
                       'lacked the satire and murder mystery premise.'},
 'tt10062292': {'description': 'After grief and temporary paralysis, impulsive Indian American '
                               'teen Devi Vishwakumar tries to reinvent her high-school life while '
                               'juggling friendship, family pressure, identity, and romance.',
                'source_url': 'https://en.wikipedia.org/wiki/Never_Have_I_Ever_(TV_series)',
                'note': 'Verified by tconst/title/year against Wikipedia: 2020 Netflix '
                        "coming-of-age series about Devi after her father's death; old description "
                        'was generic.'},
 'tt8178634': {'description': 'A fictionalized epic imagines revolutionaries Komaram Bheem and '
                              'Alluri Sitarama Raju forming a fierce friendship before rising '
                              'against British colonial power through spectacle, sacrifice, and '
                              'revolt.',
               'source_url': 'https://en.wikipedia.org/wiki/RRR',
               'note': 'Verified by tconst/title/year against Wikipedia: 2022 S. S. Rajamouli film '
                       'about fictionalized versions of Bheem and Raju; old description was too '
                       'skeletal.'},
 'tt1029234': {'description': 'A traumatized survivor seeks revenge on the family she believes '
                              'tortured her, but the attack exposes a secret society obsessed with '
                              'suffering, martyrdom, and the afterlife.',
               'source_url': 'https://en.wikipedia.org/wiki/Martyrs_(2008_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2008 Pascal Laugier '
                       'horror film about Lucie, Anna, and a torture cult; old description was '
                       'vague and incomplete.'},
 'tt4508902': {'description': 'Saitama can defeat any enemy with one punch, leaving him bored and '
                              'alienated as he joins the Hero Association and mentors cyborg Genos '
                              'amid absurd monster attacks.',
               'source_url': 'https://en.wikipedia.org/wiki/One-Punch_Man',
               'note': 'Verified by tconst/title/year against Wikipedia: 2015 anime adaptation of '
                       'One-Punch Man about Saitama and Genos; old description was accurate but '
                       'thin.'},
 'tt4729430': {'description': 'A spoiled postal academy failure is sent to a frozen feuding town, '
                              'where his friendship with reclusive toymaker Klaus slowly creates '
                              'kindness, community, and the Santa Claus legend.',
               'source_url': 'https://en.wikipedia.org/wiki/Klaus_(film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 animated origin '
                       'story about Jesper, Smeerensburg, and Klaus; old description was related '
                       'but too minimal.'},
 'tt9243946': {'description': 'After escaping captivity, Jesse Pinkman hides from police and old '
                              'enemies while seeking money, a new identity, and a fragile chance '
                              'at freedom after Breaking Bad.',
               'source_url': 'https://en.wikipedia.org/wiki/El_Camino:_A_Breaking_Bad_Movie',
               'note': 'Verified by tconst/title/year against Wikipedia: 2019 Breaking Bad sequel '
                       'following Jesse after the finale; old description lacked the '
                       'escape-survival premise.'},
 'tt9174558': {'description': "Interwoven stories trace OxyContin's rise, Purdue Pharma's "
                              'marketing, devastated patients and families, and the legal fight '
                              "around America's opioid crisis.",
               'source_url': 'https://en.wikipedia.org/wiki/Dopesick_(miniseries)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2021 miniseries about '
                       'Purdue Pharma, OxyContin, affected communities, and legal investigations; '
                       'old description was very broad.'},
 'tt14403178': {'description': 'A road-rage clash between struggling contractor Danny and '
                               'successful entrepreneur Amy spirals into an obsessive feud, '
                               'exposing loneliness, class resentment, anger, and '
                               'self-destruction.',
                'source_url': 'https://en.wikipedia.org/wiki/Beef_(TV_series)',
                'note': 'Verified by tconst/title/year against Wikipedia: 2023 Netflix series '
                        'about Danny Cho and Amy Lau after a road-rage incident; old description '
                        'was accurate but underspecified.'},
 'tt9179430': {'description': 'A black-ops investigator probes masked vigilante killings and '
                              'uncovers a hidden war involving a former commander, a drug '
                              'syndicate, grief, revenge, and buried identities.',
               'source_url': 'https://en.wikipedia.org/wiki/Vikram_(2022_film)',
               'note': 'Verified by tconst/title/year against Wikipedia: 2022 Tamil neo-noir '
                       "action thriller involving Amar, Karnan/Vikram, and Sandhanam's drug "
                       'syndicate; old description was too generic.'},
 'tt0156887': {'description': 'Former pop idol Mima becomes an actress and is stalked by an '
                              'obsessive fan and a phantom online persona, blurring celebrity '
                              'pressure, identity, paranoia, and psychological horror.',
               'source_url': 'https://en.wikipedia.org/wiki/Perfect_Blue',
               'note': 'Verified by tconst/title/year against Wikipedia: 1997 Satoshi Kon thriller '
                       'about Mima leaving CHAM! and losing her grip on reality; old description '
                       'was incomplete.'},
 'tt2303687': {'description': 'Anti-Corruption Unit 12 investigates police officers suspected of '
                              'misconduct, uncovering institutional rot, organized-crime links, '
                              'loyalty tests, and procedural tension inside the force.',
               'source_url': 'https://en.wikipedia.org/wiki/Line_of_Duty',
               'note': 'Verified by tconst/title/year against Wikipedia: 2012 BBC series about '
                       'AC-12 policing corrupt police; old description covered only the initial '
                       'setup.'},
 'tt0770828': {'description': 'Clark Kent wrestles with his alien origins and human upbringing as '
                              'General Zod threatens Earth, forcing him to become Superman amid '
                              'identity, sacrifice, and large-scale destruction.',
               'source_url': 'https://en.wikipedia.org/wiki/Man_of_Steel_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2013 Zack '
                       'Snyder Superman film; old description was unrelated to the sourced '
                       'premise.'},
 'tt0988045': {'description': 'Sherlock Holmes and Dr. Watson investigate Lord Blackwood, whose '
                              'staged supernatural return masks a political conspiracy in a fast, '
                              'combative Victorian mystery adventure.',
               'source_url': 'https://en.wikipedia.org/wiki/Sherlock_Holmes_(2009_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2009 Guy '
                       'Ritchie Sherlock Holmes film; old description was overly thin.'},
 'tt1343092': {'description': "Nick Carraway is drawn into Jay Gatsby's lavish Long Island world, "
                              'where obsession with Daisy, wealth, illusion, and class anxiety '
                              'spiral into tragedy.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Great_Gatsby_(2013_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2013 Baz '
                       'Luhrmann adaptation of F. Scott Fitzgerald; old description was too '
                       'shallow for recommendation features.'},
 'tt0493464': {'description': 'An anxious office worker learns his murdered father belonged to a '
                              'secret assassin fraternity, entering a hyperviolent world of '
                              'destiny, betrayal, and revenge.',
               'source_url': 'https://en.wikipedia.org/wiki/Wanted_(2008_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2008 Timur '
                       'Bekmambetov action thriller; old Western same-title description was '
                       'unrelated.'},
 'tt1193138': {'description': 'A corporate downsizing expert devoted to airports, status miles, '
                              'and emotional detachment has his rootless life challenged by '
                              'romance and a young efficiency consultant.',
               'source_url': 'https://en.wikipedia.org/wiki/Up_in_the_Air_(2009_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2009 Jason '
                       'Reitman comedy-drama; old radio-murder description was unrelated.'},
 'tt2788710': {'description': 'A celebrity tabloid host and his producer land an interview with '
                              "North Korea's leader, then are recruited by the CIA for a reckless "
                              'assassination mission.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Interview',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2014 '
                       'political satire action comedy; old documentary-serial-killer description '
                       'was unrelated.'},
 'tt2140479': {'description': 'A brilliant autistic accountant secretly works for criminal clients '
                              "while uncooking a corporation's books, drawing assassins, federal "
                              'agents, and buried family trauma toward him.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Accountant_(2016_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2016 Ben '
                       'Affleck action thriller; old farmer/accountant description was unrelated.'},
 'tt0278504': {'description': 'A sleep-deprived Los Angeles detective investigating an Alaska '
                              'murder is blackmailed by the killer after a foggy shooting, turning '
                              'guilt and exhaustion into psychological noir.',
               'source_url': 'https://en.wikipedia.org/wiki/Insomnia_(2002_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2002 '
                       'Christopher Nolan thriller; old insomnia description was vague and not '
                       'source-useful.'},
 'tt0212985': {'description': 'Years after Buffalo Bill, Clarice Starling is pulled back toward '
                              'Hannibal Lecter as a disfigured survivor plots revenge against the '
                              'brilliant, cannibalistic psychiatrist.',
               'source_url': 'https://en.wikipedia.org/wiki/Hannibal_(2001_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2001 Ridley '
                       'Scott psychological horror sequel; old ancient-general description was '
                       'unrelated.'},
 'tt4263482': {'description': 'In 1630s New England, a banished Puritan family unravels beside a '
                              'dark forest as religious terror, suspicion, and possible witchcraft '
                              'consume them.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Witch_(2015_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2015 Robert '
                       'Eggers folk horror; old fantasy-troubadour description was unrelated.'},
 'tt0429493': {'description': 'Framed for a military crime they did not commit, a rogue '
                              'special-ops team escapes custody and stages elaborate missions to '
                              'clear its name.',
               'source_url': 'https://en.wikipedia.org/wiki/The_A-Team_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2010 Joe '
                       'Carnahan film adaptation of The A-Team; old TV-episode style description '
                       'was unrelated.'},
 'tt1212450': {'description': 'In Prohibition-era Virginia, the Bondurant brothers defend their '
                              'moonshine empire against a corrupt special deputy in a bloody '
                              'Southern Gothic gangster story.',
               'source_url': 'https://en.wikipedia.org/wiki/Lawless_(2012_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2012 John '
                       'Hillcoat crime drama; old John Lawless description was unrelated.'},
 'tt7959026': {'description': 'An elderly horticulturist becomes a drug courier for a Mexican '
                              'cartel, risking prison and family estrangement while confronting '
                              'regret late in life.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Mule_(2018_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2018 Clint '
                       'Eastwood crime drama; old Spanish Civil War mule description was '
                       'unrelated.'},
 'tt1315981': {'description': 'A grieving gay professor in 1960s Los Angeles moves through one '
                              "carefully planned day after his partner's death, confronting "
                              'loneliness, beauty, and despair.',
               'source_url': 'https://en.wikipedia.org/wiki/A_Single_Man',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2009 Tom '
                       'Ford drama; old flapper description was unrelated.'},
 'tt2980592': {'description': "A polite ex-soldier visits a fallen comrade's family, but his charm "
                              'hides lethal programming and brings violent secrets to their home.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Guest_(2014_American_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2014 Adam '
                       'Wingard action thriller; old servant-boy description was unrelated.'},
 'tt4218572': {'description': 'After their criminal husbands die in a botched robbery, four '
                              'Chicago widows plan a heist of their own amid political corruption, '
                              'grief, and survival pressure.',
               'source_url': 'https://en.wikipedia.org/wiki/Widows_(2018_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2018 Steve '
                       'McQueen neo-noir heist thriller; old cast/plot details were from a '
                       'different Widows.'},
 'tt7456310': {'description': 'A young Russian model is secretly trained as a KGB assassin, '
                              'navigating shifting loyalties, double crosses, and brutal missions '
                              'while trying to win freedom.',
               'source_url': 'https://en.wikipedia.org/wiki/Anna_(2019_feature_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2019 Luc '
                       'Besson action thriller; old enslaved-woman description was unrelated.'},
 'tt6977338': {'description': 'Three sheltered sixth-grade boys skip school and stumble through '
                              'drones, drugs, parties, and panic while trying to learn how to kiss '
                              "before a classmate's party.",
               'source_url': 'https://en.wikipedia.org/wiki/Good_Boys_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2019 '
                       'coming-of-age comedy; old rent-boys description was unrelated and unsafe.'},
 'tt10579952': {'description': 'An alcoholic professor sent to a juvenile detention school clashes '
                               'with a ruthless gangster exploiting the students, turning '
                               'mentorship into violent redemption.',
                'source_url': 'https://en.wikipedia.org/wiki/Master_(2021_film)',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 Tamil '
                        'action drama starring Vijay; old Qi Gong master description was '
                        'unrelated.'},
 'tt1220719': {'description': 'Wing Chun master Ip Man tries to protect dignity and community '
                              'during the Japanese occupation of Foshan, balancing restraint, '
                              'martial honor, and wartime brutality.',
               'source_url': 'https://en.wikipedia.org/wiki/Ip_Man_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2008 Hong '
                       'Kong biographical martial arts film; old description was accurate but too '
                       'thin.'},
 'tt10280296': {'description': 'After surviving trauma and exile, revolutionary Sardar Udham Singh '
                               'seeks justice for the Jallianwala Bagh massacre by assassinating '
                               "Michael O'Dwyer in London.",
                'source_url': 'https://en.wikipedia.org/wiki/Sardar_Udham',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 '
                        'biographical historical drama; old description was only a name label.'},
 'tt1883092': {'description': 'Five German friends enter World War II with youthful hope, then are '
                              'fractured by combat, occupation, ideology, betrayal, and guilt '
                              'between 1941 and 1945.',
               'source_url': 'https://en.wikipedia.org/wiki/Generation_War',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2013 German '
                       'WWII miniseries; old description was too skeletal.'},
 'tt11000902': {'description': 'Gentle aristocrat Stede Bonnet abandons privilege to become a '
                               'pirate captain, finding absurd danger, queer romance, and '
                               'self-invention alongside Blackbeard.',
                'source_url': 'https://en.wikipedia.org/wiki/Our_Flag_Means_Death',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2022 period '
                        'romantic comedy series; old description was thin.'},
 'tt2356180': {'description': 'Indian runner Milkha Singh rises from Partition trauma and personal '
                              'loss to athletic greatness, chasing discipline, national pride, and '
                              'redemption on the track.',
               'source_url': 'https://en.wikipedia.org/wiki/Bhaag_Milkha_Bhaag',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2013 '
                       'biographical sports drama about Milkha Singh; old description overfocused '
                       'on one Olympic result.'},
 'tt0795176': {'description': "A sweeping BBC nature documentary explores Earth's habitats, "
                              'wildlife behavior, climate extremes, and ecological wonder through '
                              'large-scale, immersive natural-history filmmaking.',
               'source_url': 'https://en.wikipedia.org/wiki/Planet_Earth_(2006_TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2006 BBC '
                       'nature documentary miniseries; old description was too minimal.'},
 'tt9253284': {'description': 'Cassian Andor moves from cynical survivor to rebel operative as the '
                              'Empire tightens control, tracing espionage, sacrifice, prison '
                              'revolt, and the birth of resistance.',
               'source_url': 'https://en.wikipedia.org/wiki/Andor',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2022 Star '
                       'Wars political spy series; old description was too terse.'},
 'tt2531336': {'description': 'Inspired by gentleman thief Arsene Lupin, Assane Diop uses '
                              'disguise, charm, and elaborate heists to avenge his father and '
                              "expose a wealthy family's corruption.",
               'source_url': 'https://en.wikipedia.org/wiki/Lupin_(French_TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 French '
                       'mystery thriller series; old description was too generic.'},
 'tt2098220': {'description': 'Gon Freecss becomes a Hunter to find his absent father, entering '
                              'dangerous exams, friendships, supernatural battles, and morally '
                              'complex adventures.',
               'source_url': 'https://en.wikipedia.org/wiki/Hunter_%C3%97_Hunter_(2011_TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2011 anime '
                       'adaptation; old description was too thin.'},
 'tt10970552': {'description': 'An au pair arrives at a haunted English manor where orphaned '
                               'children, buried romances, and lingering spirits turn grief into '
                               'gothic supernatural tragedy.',
                'source_url': 'https://en.wikipedia.org/wiki/The_Haunting_of_Bly_Manor',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 Mike '
                        'Flanagan gothic romance horror miniseries; old description was too thin.'},
 'tt1831164': {'description': 'A surreal Istanbul comedy follows lovestruck Mecnun, reincarnated '
                              'Leyla, and eccentric friends through absurd quests, magical '
                              'realism, family chaos, and romantic longing.',
               'source_url': 'https://en.wikipedia.org/wiki/Leyla_and_Mecnun',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Turkish '
                       'surreal absurd comedy series; old description was bland.'},
 'tt2347569': {'description': "A drifting New York dancer loses her best friend's daily intimacy "
                              'and stumbles through apartments, work, ambition, and identity in a '
                              'bittersweet quarter-life comedy.',
               'source_url': 'https://en.wikipedia.org/wiki/Frances_Ha',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2012 Noah '
                       'Baumbach/Greta Gerwig comedy-drama; old description misframed the '
                       'emotional focus.'},
 'tt0476735': {'description': 'After the 1980 Turkish coup, a leftist journalist returns to his '
                              'estranged rural family with his young son, reopening political '
                              'wounds and generational grief.',
               'source_url': 'https://en.wikipedia.org/wiki/My_Father_and_My_Son',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2005 Cagan '
                       'Irmak family drama; old description was too thin.'},
 'tt11464826': {'description': 'Former tech insiders expose how social-media platforms exploit '
                               'attention, polarization, addiction, and surveillance capitalism, '
                               'blending documentary testimony with dramatized consequences.',
                'source_url': 'https://en.wikipedia.org/wiki/The_Social_Dilemma',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 '
                        'docudrama about social media harms; old description was too broad.'},
 'tt3203606': {'description': 'Blacklisted screenwriter Dalton Trumbo battles Hollywood repression '
                              'and political fear during the Red Scare, secretly writing films '
                              'while fighting for credit and dignity.',
               'source_url': 'https://en.wikipedia.org/wiki/Trumbo_(2015_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2015 Jay '
                       'Roach biographical drama; old description was accurate but thin.'},
 'tt8740976': {'description': 'A journalist investigates Anna Sorokin, the fake German heiress who '
                              'conned New York elites, while ambition, class performance, and '
                              'media fascination blur truth.',
               'source_url': 'https://en.wikipedia.org/wiki/Inventing_Anna',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2022 Shonda '
                       'Rhimes miniseries; old description was serviceable but less useful for '
                       'mood features.'},
 'tt10813940': {'description': 'Free-spirited Georgia and teenage Ginny try to build a stable life '
                               'in New England, but secrets, romance, class tension, and family '
                               'trauma keep surfacing.',
                'source_url': 'https://en.wikipedia.org/wiki/Ginny_%26_Georgia',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 '
                        'Netflix comedy-drama; old description was thin.'},
 'tt5700672': {'description': 'A workaholic father and his young daughter board a train to Busan '
                              'as a zombie outbreak erupts, forcing passengers into frantic moral '
                              'and physical survival.',
               'source_url': 'https://en.wikipedia.org/wiki/Train_to_Busan',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2016 Yeon '
                       'Sang-ho zombie thriller; expanded for conflict, tone, and premise.'},
 'tt9770150': {'description': 'After losing work and home in a company-town collapse, Fern lives '
                              'as a modern nomad across the American West, finding grief, '
                              'resilience, solitude, and fragile community.',
               'source_url': 'https://en.wikipedia.org/wiki/Nomadland',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 Chloe '
                       'Zhao drama; old description was accurate but thin.'},
 'tt7395114': {'description': 'Across postwar Ohio and West Virginia, damaged families, corrupt '
                              'faith, serial violence, and inherited trauma converge around a '
                              'young man trying to protect what remains.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Devil_All_the_Time_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 crime '
                       'drama; expanded from a generic corruption summary.'},
 'tt5742374': {'description': 'A traumatized veteran who rescues trafficked girls searches for a '
                              'kidnapped teen, only to uncover political rot while violence and '
                              'suicidal despair close in.',
               'source_url': 'https://en.wikipedia.org/wiki/You_Were_Never_Really_Here',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2017 Lynne '
                       'Ramsay crime drama; expanded for mood and conflict.'},
 'tt7939766': {'description': "A young woman visits her boyfriend's isolated family farm as "
                              'reality, memory, and identity begin to fracture in a wintry '
                              'psychological descent.',
               'source_url': 'https://en.wikipedia.org/wiki/I%27m_Thinking_of_Ending_Things',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 Charlie '
                       'Kaufman psychological thriller; old description was too minimal.'},
 'tt4430212': {'description': 'A protective family man constructs an elaborate cover-up after a '
                              'crime threatens his loved ones, turning movie knowledge, police '
                              'pressure, and moral compromise into suspense.',
               'source_url': 'https://en.wikipedia.org/wiki/Drishyam_(2015_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2015 Hindi '
                       'thriller; expanded from a generic law/family line.'},
 'tt9815454': {'description': 'A young Hasidic woman escapes an arranged marriage in Brooklyn for '
                              'Berlin, pursuing music and autonomy while her past, faith, and '
                              'community obligations follow.',
               'source_url': 'https://en.wikipedia.org/wiki/Unorthodox_(miniseries)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 '
                       'miniseries; expanded for identity and conflict.'},
 'tt11823076': {'description': "A true-crime docuseries dives into America's exotic-animal "
                               'underworld, where Joe Exotic, Carole Baskin, rivalry, '
                               'exploitation, ego, and criminal schemes collide.',
                'source_url': 'https://en.wikipedia.org/wiki/Tiger_King',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Tiger King '
                        'miniseries; old description was too broad.'},
 'tt6473300': {'description': "In lawless Mirzapur, two brothers are pulled into a crime lord's "
                              'empire, where family ambition, revenge, guns, and political power '
                              'consume everyone nearby.',
               'source_url': 'https://en.wikipedia.org/wiki/Mirzapur_(TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Indian crime '
                       'series; old description was generic.'},
 'tt4574334': {'description': 'In 1980s Hawkins, children, families, and a telekinetic girl '
                              'confront secret experiments, alternate-dimension monsters, '
                              'friendship, grief, and small-town supernatural terror.',
               'source_url': 'https://en.wikipedia.org/wiki/Stranger_Things',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Stranger '
                       'Things; expanded for mood and ensemble conflict.'},
 'tt1675434': {'description': 'A wealthy quadriplegic aristocrat hires an irreverent ex-con as his '
                              'caregiver, forming a funny, tender friendship across class, '
                              'disability, grief, and freedom.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Intouchables',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2011 French '
                       'buddy comedy-drama; expanded from a basic friendship line.'},
 'tt2356777': {'description': 'An anthology of haunted investigations follows detectives through '
                              'murder, obsession, occult dread, corruption, and moral decay, with '
                              'each season exploring a different crime.',
               'source_url': 'https://en.wikipedia.org/wiki/True_Detective',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: True '
                       'Detective anthology; old description was too generic.'},
 'tt8111088': {'description': 'A lone Mandalorian bounty hunter protects a mysterious '
                              'Force-sensitive child while crossing the lawless Star Wars frontier '
                              'of mercenaries, remnants, honor, and found family.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Mandalorian',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Star Wars '
                       'space western series; expanded for core relationship and tone.'},
 'tt6723592': {'description': 'A CIA operative learns to manipulate time inversion while chasing a '
                              'future-linked threat, turning espionage into a puzzle of entropy, '
                              'fate, and global annihilation.',
               'source_url': 'https://en.wikipedia.org/wiki/Tenet',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 '
                       'Christopher Nolan sci-fi thriller; expanded from a generic mission line.'},
 'tt10919420': {'description': "Debt-ridden contestants enter deadly versions of children's games "
                               'for a fortune, exposing desperation, class cruelty, betrayal, and '
                               'survival under bright candy-colored horror.',
                'source_url': 'https://en.wikipedia.org/wiki/Squid_Game',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 Korean '
                        'survival drama; expanded for tone and themes.'},
 'tt3581920': {'description': 'In a fungal post-apocalypse, hardened smuggler Joel escorts immune '
                              'teenager Ellie across America, where infection, violence, grief, '
                              'and reluctant love reshape them.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Last_of_Us_(TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2023 HBO '
                       'adaptation; expanded from a generic trek line.'},
 'tt0119698': {'description': 'Cursed prince Ashitaka becomes caught between forest spirits, '
                              'warrior San, and an ironworks consuming nature, in an epic conflict '
                              'over hatred, survival, and coexistence.',
               'source_url': 'https://en.wikipedia.org/wiki/Princess_Mononoke',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 1997 '
                       'Miyazaki anime; expanded for conflict and themes.'},
 'tt0877057': {'description': 'A brilliant student uses a supernatural notebook to kill criminals '
                              'and remake society, sparking a tense cat-and-mouse battle with '
                              'detective L over justice and ego.',
               'source_url': 'https://en.wikipedia.org/wiki/Death_Note_(2006_TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Death Note '
                       'anime; expanded from a one-line premise.'},
 'tt9140560': {'description': 'Wanda and Vision live through shifting sitcom realities in a sealed '
                              'suburban town, where grief, denial, magic, and superhero mystery '
                              'distort domestic fantasy.',
               'source_url': 'https://en.wikipedia.org/wiki/WandaVision',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 Marvel '
                       'miniseries; expanded for mood and conflict.'},
 'tt1028532': {'description': 'A professor forms a devoted bond with an Akita whose daily loyalty '
                              'at a train station becomes a quiet story of love, loss, and '
                              'remembrance.',
               'source_url': 'https://en.wikipedia.org/wiki/Hachi:_A_Dog%27s_Tale',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2009 Lasse '
                       'Hallstrom drama; old description was thin.'},
 'tt5311514': {'description': 'Two teenagers mysteriously swap bodies across distance and time, '
                              'turning comic confusion into longing, disaster, memory, and a '
                              'desperate search for connection.',
               'source_url': 'https://en.wikipedia.org/wiki/Your_Name',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2016 Makoto '
                       'Shinkai anime; expanded for emotional stakes.'},
 'tt3464902': {'description': 'In a deadpan dystopia, single people must find partners or become '
                              'animals, forcing one lonely man through absurd rituals, rebellion, '
                              'and warped romance.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Lobster',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2015 Yorgos '
                       'Lanthimos film; expanded for tone.'},
 'tt8228288': {'description': 'Inside a vertical prison where food descends floor by floor, '
                              'inequality becomes literal as hunger, brutality, solidarity, and '
                              'moral collapse intensify.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Platform_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2019 Spanish '
                       'social sci-fi horror; expanded for themes.'},
 'tt10234724': {'description': 'Museum worker Steven Grant discovers he shares a body with '
                               'mercenary Marc Spector, drawing them into Egyptian gods, '
                               'dissociation, trauma, and supernatural vigilantism.',
                'source_url': 'https://en.wikipedia.org/wiki/Moon_Knight_(miniseries)',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2022 Marvel '
                        'miniseries; expanded for psychological premise.'},
 'tt11126994': {'description': 'In divided Piltover and Zaun, sisters Vi and Jinx are torn apart '
                               'by class conflict, magic technology, crime, trauma, and political '
                               'unrest.',
                'source_url': 'https://en.wikipedia.org/wiki/Arcane_(TV_series)',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Arcane '
                        'animated series; expanded from a generic origins line.'},
 'tt11138512': {'description': "A Viking prince survives his father's murder and spends his life "
                               'pursuing revenge, destiny, and brutal mythic justice across a '
                               'harsh Norse world.',
                'source_url': 'https://en.wikipedia.org/wiki/The_Northman',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2022 Robert '
                        'Eggers epic; expanded for mood and premise.'},
 'tt3011894': {'description': 'Six darkly comic stories of revenge, humiliation, corruption, and '
                              'sudden violence push ordinary people past civilized limits into '
                              'explosive moral chaos.',
               'source_url': 'https://en.wikipedia.org/wiki/Wild_Tales_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2014 '
                       'Argentine anthology; expanded from one segment summary.'},
 'tt2338151': {'description': 'An innocent alien stranded on Earth questions religion, rituals, '
                              'and social hypocrisy while searching for his stolen communication '
                              'device and discovering human love.',
               'source_url': 'https://en.wikipedia.org/wiki/PK_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2014 '
                       'Rajkumar Hirani satire; expanded for premise and themes.'},
 'tt0756683': {'description': 'During a farewell gathering, a professor claims he has lived for '
                              '14,000 years, forcing colleagues into a philosophical debate about '
                              'history, faith, and identity.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Man_from_Earth',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2007 sci-fi '
                       'drama; expanded from short discussion premise.'},
 'tt1038988': {'description': 'A TV reporter and cameraman are trapped inside a quarantined '
                              'Barcelona apartment building as an infection spreads, turning '
                              'found-footage realism into claustrophobic horror.',
               'source_url': 'https://en.wikipedia.org/wiki/Rec_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2007 Spanish '
                       'horror film; expanded for setting and tone.'},
 'tt0200465': {'description': 'A 1971 London bank robbery uncovers compromising secrets, police '
                              'corruption, and political scandal, turning a heist into a dangerous '
                              'cover-up thriller.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Bank_Job',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2008 Roger '
                       'Donaldson heist thriller; old description was vague and partly '
                       'misleading.'},
 'tt10288566': {'description': 'Four Danish teachers test whether steady drinking improves their '
                               'lives, but the experiment shifts from comic liberation to '
                               'addiction, denial, grief, and reckoning.',
                'source_url': 'https://en.wikipedia.org/wiki/Another_Round',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 Thomas '
                        'Vinterberg drama; expanded for emotional arc.'},
 'tt10155688': {'description': 'Small-town detective Mare Sheehan investigates a local murder '
                               'while family grief, community secrets, addiction, and personal '
                               'collapse press in around her.',
                'source_url': 'https://en.wikipedia.org/wiki/Mare_of_Easttown',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 crime '
                        'drama miniseries; expanded from brief murder line.'},
 'tt6412452': {'description': 'Six Coen brothers Western tales move from absurd gunfighter comedy '
                              'to bleak frontier mortality, exploring violence, chance, '
                              'loneliness, performance, and death.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Ballad_of_Buster_Scruggs',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2018 '
                       'anthology Western; expanded for tone and themes.'},
 'tt6718170': {'description': 'Brooklyn plumbers Mario and Luigi are pulled into the Mushroom '
                              'Kingdom, where Mario teams with Peach to stop Bowser and rescue his '
                              'brother.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Super_Mario_Bros._Movie',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2023 '
                       'animated film; expanded for setup and stakes.'},
 'tt9288030': {'description': 'Drifter and former military investigator Jack Reacher is framed for '
                              'murder in a small Georgia town, exposing corruption through blunt '
                              'detective work and bone-crunching action.',
               'source_url': 'https://en.wikipedia.org/wiki/Reacher_(TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Reacher '
                       'series; expanded for tone and premise.'},
 'tt0290673': {'description': 'Told in reverse chronology, a brutal assault and revenge spiral '
                              'through Paris nightclubs, forcing viewers into rage, trauma, time, '
                              'and moral horror.',
               'source_url': 'https://en.wikipedia.org/wiki/Irr%C3%A9versible',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2002 Gaspar '
                       'Noe film; expanded for structure and tone.'},
 'tt2582496': {'description': 'A self-conscious teen filmmaker befriends a classmate with '
                              'leukemia, making awkward parody films while learning empathy, '
                              'grief, and the limits of performative kindness.',
               'source_url': 'https://en.wikipedia.org/wiki/Me_and_Earl_and_the_Dying_Girl_(film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2015 '
                       'coming-of-age drama; expanded from friendship line.'},
 'tt9495224': {'description': 'In an interactive Black Mirror story, a young programmer adapting a '
                              'fantasy novel in 1984 becomes trapped in branching choices, '
                              'paranoia, control, and fractured reality.',
               'source_url': 'https://en.wikipedia.org/wiki/Black_Mirror:_Bandersnatch',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2018 '
                       'interactive film; expanded for format and mood.'},
 'tt2884206': {'description': 'A molecular biologist studying eye patterns confronts love, loss, '
                              'reincarnation, and the limits of scientific certainty after '
                              'discoveries challenge his worldview.',
               'source_url': 'https://en.wikipedia.org/wiki/I_Origins',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2014 Mike '
                       'Cahill drama; expanded from abstract premise.'},
 'tt3289956': {'description': "Father-and-son coroners examine an unidentified woman's body "
                              'overnight, uncovering impossible injuries and occult clues as the '
                              'morgue becomes a tightening supernatural trap.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Autopsy_of_Jane_Doe',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2016 Andre '
                       'Ovredal horror film; expanded for atmosphere and conflict.'},
 'tt1478964': {'description': 'A South London teen gang and their neighbors defend a council '
                              'estate from savage alien invaders, mixing creature-feature action '
                              'with streetwise comedy and class tension.',
               'source_url': 'https://en.wikipedia.org/wiki/Attack_the_Block',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2011 Joe '
                       'Cornish sci-fi comedy; expanded from a terse premise.'},
 'tt5971474': {'description': 'Restless mermaid Ariel trades her voice for legs after falling for '
                              "Prince Eric, challenging her father, Ursula's bargain, and the "
                              'boundary between sea and human worlds.',
               'source_url': 'https://en.wikipedia.org/wiki/The_Little_Mermaid_(2023_film)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2023 Rob '
                       'Marshall live-action musical; old sea-foam description belonged to a '
                       'different telling.'},
 'tt9389998': {'description': 'Ambitious laborer Pushpa Raj rises through a red sandalwood '
                              'smuggling syndicate, clashing with police, rivals, and class '
                              'contempt in a gritty crime action drama.',
               'source_url': 'https://en.wikipedia.org/wiki/Pushpa:_The_Rise',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2021 Sukumar '
                       'action crime film; expanded from a generic smuggling line.'},
 'tt12343534': {'description': 'Yuji Itadori swallows a cursed object to save others and joins '
                               'jujutsu sorcerers fighting grotesque curses while sharing his body '
                               'with a deadly demon.',
                'source_url': 'https://en.wikipedia.org/wiki/Jujutsu_Kaisen_(TV_series)',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Jujutsu '
                        'Kaisen anime; expanded for premise and stakes.'},
 'tt12411074': {'description': 'Skeptical hematologist Refaat Ismail is dragged into supernatural '
                               'cases in 1960s Egypt, confronting old love, scientific doubt, '
                               'folklore, and buried fear.',
                'source_url': 'https://en.wikipedia.org/wiki/Paranormal_(TV_series)',
                'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: Egyptian '
                        'horror fantasy series; expanded for mood and setting.'},
 'tt9612516': {'description': 'A decorated general and a team of scientists, politicians, and '
                              'misfits try to launch the U.S. Space Force, turning bureaucracy and '
                              'ambition into workplace satire.',
               'source_url': 'https://en.wikipedia.org/wiki/Space_Force_(TV_series)',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2020 '
                       'workplace comedy series; old description was too generic.'},
 'tt0280249': {'description': 'Monkey-tailed Goku meets Bulma and begins a comic martial-arts '
                              'quest for the seven Dragon Balls, encountering rivals, monsters, '
                              'training, and adventure.',
               'source_url': 'https://en.wikipedia.org/wiki/Dragon_Ball_(TV_series)',
               'note': 'Quality refinement verified by title/year/description and Wikipedia: '
                       'original Dragon Ball TV series; old description was misspelled and too '
                       'thin.'},
 'tt3262342': {'description': "In the world's first fully painted feature film, Armand Roulin "
                              "investigates Vincent van Gogh's final days, turning an art mystery "
                              'into a portrait of grief and memory.',
               'source_url': 'https://en.wikipedia.org/wiki/Loving_Vincent',
               'note': 'Quality refinement verified by tconst via Wikidata/Wikipedia: 2017 '
                       'animated biographical mystery; expanded from a short death-mystery line.'}}


VERIFIED_DESCRIPTION_OK = {'tt10048342': {'source_url': 'https://en.wikipedia.org/wiki/The_Queen%27s_Gambit_(miniseries)',
                'note': "Verified by IMDb tconst via Wikidata/Wikipedia as The Queen's Gambit "
                        '(2020); current description is supported by the sourced premise, so no '
                        'correction was applied.'},
 'tt1187043': {'source_url': 'https://en.wikipedia.org/wiki/3_Idiots',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as 3 Idiots (2009); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt1706620': {'source_url': 'https://en.wikipedia.org/wiki/Snowpiercer',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Snowpiercer (2013); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt7784604': {'source_url': 'https://en.wikipedia.org/wiki/Hereditary_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Hereditary (2018); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt0399201': {'source_url': 'https://en.wikipedia.org/wiki/The_Island_(2005_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Island (2005); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1591095': {'source_url': 'https://en.wikipedia.org/wiki/Insidious_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Insidious (2010); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4925292': {'source_url': 'https://en.wikipedia.org/wiki/Lady_Bird_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Lady Bird (2017); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5727208': {'source_url': 'https://en.wikipedia.org/wiki/Uncut_Gems',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Uncut Gems (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2397535': {'source_url': 'https://en.wikipedia.org/wiki/Predestination_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Predestination (2014); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt10986410': {'source_url': 'https://en.wikipedia.org/wiki/Ted_Lasso',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Ted Lasso (2020); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt1817273': {'source_url': 'https://en.wikipedia.org/wiki/The_Place_Beyond_the_Pines',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Place Beyond the '
                       'Pines (2012); current description is supported by the sourced premise, so '
                       'no correction was applied.'},
 'tt0813715': {'source_url': 'https://en.wikipedia.org/wiki/Heroes_(American_TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Heroes (2006); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt0485947': {'source_url': 'https://en.wikipedia.org/wiki/Mr._Nobody_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Mr. Nobody (2009); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2321549': {'source_url': 'https://en.wikipedia.org/wiki/The_Babadook',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Babadook (2014); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt10954984': {'source_url': 'https://en.wikipedia.org/wiki/Nope_(film)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Nope (2022); current '
                        'description is supported by the sourced premise, so no correction was '
                        'applied.'},
 'tt2463208': {'source_url': 'https://en.wikipedia.org/wiki/The_Adam_Project',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Adam Project (2022); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt8466564': {'source_url': 'https://en.wikipedia.org/wiki/Obi-Wan_Kenobi_(miniseries)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Obi-Wan Kenobi (2022); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt15097216': {'source_url': 'https://en.wikipedia.org/wiki/Jai_Bhim_(film)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Jai Bhim (2021); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt0986264': {'source_url': 'https://en.wikipedia.org/wiki/Taare_Zameen_Par',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Taare Zameen Par (2007); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5074352': {'source_url': 'https://en.wikipedia.org/wiki/Dangal_(2016_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Dangal (2016); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt0810819': {'source_url': 'https://en.wikipedia.org/wiki/The_Danish_Girl_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Danish Girl (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt3416742': {'source_url': 'https://en.wikipedia.org/wiki/What_We_Do_in_the_Shadows',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as What We Do in the '
                       'Shadows (2014); current description is supported by the sourced premise, '
                       'so no correction was applied.'},
 'tt1465522': {'source_url': 'https://en.wikipedia.org/wiki/Tucker_%26_Dale_vs._Evil',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Tucker & Dale vs. Evil '
                       '(2010); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt4857264': {'source_url': 'https://en.wikipedia.org/wiki/The_Invisible_Guest',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Invisible Guest '
                       '(2016); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt3521126': {'source_url': 'https://en.wikipedia.org/wiki/The_Disaster_Artist_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Disaster Artist '
                       '(2017); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt1714206': {'source_url': 'https://en.wikipedia.org/wiki/The_Spectacular_Now',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Spectacular Now '
                       '(2013); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt2692904': {'source_url': 'https://en.wikipedia.org/wiki/Locke_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Locke (2013); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt1615147': {'source_url': 'https://en.wikipedia.org/wiki/Margin_Call',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Margin Call (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt8420184': {'source_url': 'https://en.wikipedia.org/wiki/The_Last_Dance_(miniseries)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Last Dance (2020); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4698684': {'source_url': 'https://en.wikipedia.org/wiki/Hunt_for_the_Wilderpeople',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Hunt for the '
                       'Wilderpeople (2016); current description is supported by the sourced '
                       'premise, so no correction was applied.'},
 'tt8110330': {'source_url': 'https://en.wikipedia.org/wiki/Dil_Bechara',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Dil Bechara (2020); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4846232': {'source_url': 'https://en.wikipedia.org/wiki/Good_Time_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Good Time (2017); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2631186': {'source_url': 'https://en.wikipedia.org/wiki/Baahubali:_The_Beginning',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Baahubali: The Beginning '
                       '(2015); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt1536537': {'source_url': 'https://en.wikipedia.org/wiki/What_Happened_to_Monday',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as What Happened to Monday '
                       '(2017); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt8404614': {'source_url': 'https://en.wikipedia.org/wiki/The_Two_Popes',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Two Popes (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1862079': {'source_url': 'https://en.wikipedia.org/wiki/Safety_Not_Guaranteed',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Safety Not Guaranteed '
                       '(2012); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt10295212': {'source_url': 'https://en.wikipedia.org/wiki/Shershaah',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Shershaah (2021); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt10574558': {'source_url': 'https://en.wikipedia.org/wiki/Midnight_Mass_(miniseries)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Midnight Mass (2021); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt1924396': {'source_url': 'https://en.wikipedia.org/wiki/The_Best_Offer',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Best Offer (2013); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2400463': {'source_url': 'https://en.wikipedia.org/wiki/The_Invitation_(2015_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Invitation (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt7462410': {'source_url': 'https://en.wikipedia.org/wiki/The_Wheel_of_Time_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Wheel of Time '
                       '(2021); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt7909970': {'source_url': 'https://en.wikipedia.org/wiki/Unbelievable_(miniseries)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Unbelievable (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1567609': {'source_url': 'https://en.wikipedia.org/wiki/Get_the_Gringo',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Get the Gringo (2012); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2494362': {'source_url': 'https://en.wikipedia.org/wiki/Bone_Tomahawk',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Bone Tomahawk (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4849438': {'source_url': 'https://en.wikipedia.org/wiki/Baahubali_2:_The_Conclusion',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Baahubali 2: The '
                       'Conclusion (2017); current description is supported by the sourced '
                       'premise, so no correction was applied.'},
 'tt0861739': {'source_url': 'https://en.wikipedia.org/wiki/Elite_Squad',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Elite Squad (2007); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2403776': {'source_url': 'https://en.wikipedia.org/wiki/Shadow_and_Bone_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Shadow and Bone (2021); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1614989': {'source_url': 'https://en.wikipedia.org/wiki/Headhunters_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Headhunters (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1954470': {'source_url': 'https://en.wikipedia.org/wiki/Gangs_of_Wasseypur',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Gangs of Wasseypur '
                       '(2012); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt9319668': {'source_url': 'https://en.wikipedia.org/wiki/1899_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as 1899 (2022); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt11743610': {'source_url': 'https://en.wikipedia.org/wiki/The_Terminal_List',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Terminal List '
                        '(2022); current description is supported by the sourced premise, so no '
                        'correction was applied.'},
 'tt8580274': {'source_url': 'https://en.wikipedia.org/wiki/Eurovision_Song_Contest:_The_Story_of_Fire_Saga',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Eurovision Song Contest: '
                       'The Story of Fire Saga (2020); current description is supported by the '
                       'sourced premise, so no correction was applied.'},
 'tt2481498': {'source_url': 'https://en.wikipedia.org/wiki/Extremely_Wicked,_Shockingly_Evil_and_Vile',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Extremely Wicked, '
                       'Shockingly Evil and Vile (2019); current description is supported by the '
                       'sourced premise, so no correction was applied.'},
 'tt1549572': {'source_url': 'https://en.wikipedia.org/wiki/Another_Earth',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Another Earth (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt8108198': {'source_url': 'https://en.wikipedia.org/wiki/Andhadhun',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Andhadhun (2018); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt3544112': {'source_url': 'https://en.wikipedia.org/wiki/Sing_Street',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Sing Street (2016); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1860242': {'source_url': 'https://en.wikipedia.org/wiki/The_Highwaymen_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Highwaymen (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt15327088': {'source_url': 'https://en.wikipedia.org/wiki/Kantara_(2022_film)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Kantara (2022); current '
                        'description is supported by the sourced premise, so no correction was '
                        'applied.'},
 'tt8267604': {'source_url': 'https://en.wikipedia.org/wiki/Capernaum_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Capernaum (2018); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4364194': {'source_url': 'https://en.wikipedia.org/wiki/The_Peanut_Butter_Falcon',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Peanut Butter Falcon '
                       '(2019); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt6292852': {'source_url': 'https://en.wikipedia.org/wiki/I_Am_Mother',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as I Am Mother (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt14208870': {'source_url': 'https://en.wikipedia.org/wiki/The_Fabelmans',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Fabelmans (2022); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt0367110': {'source_url': 'https://en.wikipedia.org/wiki/Swades',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Swades (2004); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt3863552': {'source_url': 'https://en.wikipedia.org/wiki/Bajrangi_Bhaijaan',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Bajrangi Bhaijaan '
                       '(2015); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt3007572': {'source_url': 'https://en.wikipedia.org/wiki/Locke_%26_Key_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Locke & Key (2020); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt4633690': {'source_url': 'https://en.wikipedia.org/wiki/Shot_Caller_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Shot Caller (2017); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2013293': {'source_url': 'https://en.wikipedia.org/wiki/The_Wind_Rises',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Wind Rises (2013); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt11337908': {'source_url': 'https://en.wikipedia.org/wiki/Maid_(miniseries)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Maid (2021); current '
                        'description is supported by the sourced premise, so no correction was '
                        'applied.'},
 'tt2205697': {'source_url': 'https://en.wikipedia.org/wiki/Stuck_in_Love_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Stuck in Love (2012); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2370248': {'source_url': 'https://en.wikipedia.org/wiki/Short_Term_12',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Short Term 12 (2013); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1289406': {'source_url': 'https://en.wikipedia.org/wiki/Harry_Brown_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Harry Brown (2009); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1020530': {'source_url': 'https://en.wikipedia.org/wiki/Eden_Lake',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Eden Lake (2008); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1456635': {'source_url': 'https://en.wikipedia.org/wiki/Goon_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Goon (2011); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt2057392': {'source_url': 'https://en.wikipedia.org/wiki/Eye_in_the_Sky_(2015_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Eye in the Sky (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt0112130': {'source_url': 'https://en.wikipedia.org/wiki/Pride_and_Prejudice_(1995_TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Pride and Prejudice '
                       '(1995); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt7134908': {'source_url': 'https://en.wikipedia.org/wiki/Elite_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Elite (2018); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt1716772': {'source_url': 'https://en.wikipedia.org/wiki/The_Inbetweeners_Movie',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Inbetweeners Movie '
                       '(2011); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt17663992': {'source_url': 'https://en.wikipedia.org/wiki/Scream_VI',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Scream VI (2023); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt1365050': {'source_url': 'https://en.wikipedia.org/wiki/Beasts_of_No_Nation_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Beasts of No Nation '
                       '(2015); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt2179116': {'source_url': 'https://en.wikipedia.org/wiki/The_Kings_of_Summer',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Kings of Summer '
                       '(2013); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt1562872': {'source_url': 'https://en.wikipedia.org/wiki/Zindagi_Na_Milegi_Dobara',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Zindagi Na Milegi Dobara '
                       '(2011); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt8075192': {'source_url': 'https://en.wikipedia.org/wiki/Shoplifters_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Shoplifters (2018); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1726669': {'source_url': 'https://en.wikipedia.org/wiki/Killer_Joe_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Killer Joe (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1280558': {'source_url': 'https://en.wikipedia.org/wiki/A_Wednesday!',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as A Wednesday! (2008); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5607976': {'source_url': 'https://en.wikipedia.org/wiki/His_Dark_Materials_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as His Dark Materials '
                       '(2019); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt10618286': {'source_url': 'https://en.wikipedia.org/wiki/Mank',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Mank (2020); current '
                        'description is supported by the sourced premise, so no correction was '
                        'applied.'},
 'tt1972571': {'source_url': 'https://en.wikipedia.org/wiki/A_Most_Wanted_Man_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as A Most Wanted Man '
                       '(2014); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt9059760': {'source_url': 'https://en.wikipedia.org/wiki/Normal_People_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Normal People (2020); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2359024': {'source_url': 'https://en.wikipedia.org/wiki/Blue_Ruin',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Blue Ruin (2013); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2937898': {'source_url': 'https://en.wikipedia.org/wiki/A_Most_Violent_Year',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as A Most Violent Year '
                       '(2014); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt5215952': {'source_url': 'https://en.wikipedia.org/wiki/The_Wailing_(2016_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Wailing (2016); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2401878': {'source_url': 'https://en.wikipedia.org/wiki/Anomalisa',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Anomalisa (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt0112870': {'source_url': 'https://en.wikipedia.org/wiki/Dilwale_Dulhania_Le_Jayenge',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Dilwale Dulhania Le '
                       'Jayenge (1995); current description is supported by the sourced premise, '
                       'so no correction was applied.'},
 'tt1343097': {'source_url': 'https://en.wikipedia.org/wiki/The_Girl_Who_Kicked_the_Hornets%27_Nest_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Girl Who Kicked the '
                       "Hornets' Nest (2009); current description is supported by the sourced "
                       'premise, so no correction was applied.'},
 'tt0370986': {'source_url': 'https://en.wikipedia.org/wiki/Mysterious_Skin',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Mysterious Skin (2004); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1763303': {'source_url': 'https://en.wikipedia.org/wiki/The_First_Time_(2012_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The First Time (2012); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5613484': {'source_url': 'https://en.wikipedia.org/wiki/Mid90s',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Mid90s (2018); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt14852808': {'source_url': 'https://en.wikipedia.org/wiki/The_Watcher_(2022_TV_series)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Watcher (2022); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt8291284': {'source_url': 'https://en.wikipedia.org/wiki/The_Peripheral_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Peripheral (2022); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1912398': {'source_url': 'https://en.wikipedia.org/wiki/God_Bless_America_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as God Bless America '
                       '(2011); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt0242519': {'source_url': 'https://en.wikipedia.org/wiki/Hera_Pheri_(2000_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Hera Pheri (2000); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt10293938': {'source_url': 'https://en.wikipedia.org/wiki/Outer_Banks_(TV_series)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Outer Banks (2020); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt14992922': {'source_url': 'https://en.wikipedia.org/wiki/The_Tinder_Swindler',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as The Tinder Swindler '
                        '(2022); current description is supported by the sourced premise, so no '
                        'correction was applied.'},
 'tt1242422': {'source_url': 'https://en.wikipedia.org/wiki/Cell_211',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Cell 211 (2009); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt9446688': {'source_url': 'https://en.wikipedia.org/wiki/I_Am_Not_Okay_with_This',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as I Am Not Okay With This '
                       '(2020); current description is supported by the sourced premise, so no '
                       'correction was applied.'},
 'tt2934286': {'source_url': 'https://en.wikipedia.org/wiki/Halo_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Halo (2022); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt0804484': {'source_url': 'https://en.wikipedia.org/wiki/Foundation_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Foundation (2021); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5461944': {'source_url': 'https://en.wikipedia.org/wiki/Hotel_Mumbai',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Hotel Mumbai (2018); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1910272': {'source_url': 'https://en.wikipedia.org/wiki/Steins;Gate_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Steins;Gate (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt3322420': {'source_url': 'https://en.wikipedia.org/wiki/Queen_(2013_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Queen (2013); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt3715320': {'source_url': 'https://en.wikipedia.org/wiki/Irrational_Man_(film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Irrational Man (2015); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt1166100': {'source_url': 'https://en.wikipedia.org/wiki/Ghajini_(2008_film)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Ghajini (2008); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt1821480': {'source_url': 'https://en.wikipedia.org/wiki/Kahaani',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Kahaani (2012); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt7423538': {'source_url': 'https://en.wikipedia.org/wiki/Ratched_(TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Ratched (2020); current '
                       'description is supported by the sourced premise, so no correction was '
                       'applied.'},
 'tt0387764': {'source_url': 'https://en.wikipedia.org/wiki/Peep_Show_(British_TV_series)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Peep Show (2003); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt12809988': {'source_url': 'https://en.wikipedia.org/wiki/Sweet_Tooth_(TV_series)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Sweet Tooth (2021); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt1439572': {'source_url': 'https://en.wikipedia.org/wiki/Perfect_Sense',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Perfect Sense (2011); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt5710514': {'source_url': 'https://en.wikipedia.org/wiki/I_Don%27t_Feel_at_Home_in_This_World_Anymore',
               'note': "Verified by IMDb tconst via Wikidata/Wikipedia as I Don't Feel at Home in "
                       'This World Anymore (2017); current description is supported by the sourced '
                       'premise, so no correction was applied.'},
 'tt0443465': {'source_url': 'https://en.wikipedia.org/wiki/Before_We_Go',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Before We Go (2014); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt20850406': {'source_url': 'https://en.wikipedia.org/wiki/Sita_Ramam',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Sita Ramam (2022); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'},
 'tt8291806': {'source_url': 'https://en.wikipedia.org/wiki/Pain_and_Glory',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Pain and Glory (2019); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt2304589': {'source_url': 'https://en.wikipedia.org/wiki/Defending_Jacob_(miniseries)',
               'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Defending Jacob (2020); '
                       'current description is supported by the sourced premise, so no correction '
                       'was applied.'},
 'tt10638036': {'source_url': 'https://en.wikipedia.org/wiki/Heartstopper_(TV_series)',
                'note': 'Verified by IMDb tconst via Wikidata/Wikipedia as Heartstopper (2022); '
                        'current description is supported by the sourced premise, so no correction '
                        'was applied.'}}


STOPWORDS = set(
    "the a an and or of to in on at for with from by into is are was were be as it its part ii iii iv v movie film show episode one two"
    .split()
)


def tokens(value: object) -> set[str]:
    return {
        token
        for token in re.findall(r"[a-z0-9]+", str(value).lower())
        if token not in STOPWORDS and len(token) > 1
    }


def title_description_overlap(row: pd.Series) -> float:
    title_terms = tokens(
        " ".join(
            str(row.get(column, ""))
            for column in ("primary_title", "display_title", "original_title")
        )
    )
    if not title_terms:
        return 0.0
    return len(title_terms & tokens(row.get("description", ""))) / len(title_terms)


def rebuild_combined_features(row: pd.Series) -> str:
    parts = [
        row.get("primary_title", ""),
        row.get("display_title", ""),
        row.get("original_title", ""),
        row.get("content_type", ""),
        str(row.get("genres", "")).replace(",", " "),
        row.get("description", ""),
    ]
    return " ".join(str(part).lower() for part in parts if pd.notna(part))


In [18]:
source_rows = []
verified_ok_rows = []
corrected_watch_df = final_df.copy()

missing_correction_tconsts = sorted(
    set(DESCRIPTION_MISMATCH_CORRECTIONS) - set(corrected_watch_df["tconst"])
)
assert not missing_correction_tconsts, f"Correction tconsts not found: {missing_correction_tconsts}"

for tconst, correction in DESCRIPTION_MISMATCH_CORRECTIONS.items():
    row_index = corrected_watch_df.index[corrected_watch_df["tconst"] == tconst][0]
    old_description = corrected_watch_df.at[row_index, "description"]
    corrected_watch_df.at[row_index, "description"] = correction["description"]
    source_rows.append({
        "tconst": tconst,
        "primary_title": corrected_watch_df.at[row_index, "primary_title"],
        "release_year": corrected_watch_df.at[row_index, "release_year"],
        "old_description": old_description,
        "corrected_description": correction["description"],
        "source_url": correction["source_url"],
        "verification_note": correction["note"],
    })

corrected_watch_df["combined_features"] = corrected_watch_df.apply(rebuild_combined_features, axis=1)

missing_verified_ok_tconsts = sorted(
    set(VERIFIED_DESCRIPTION_OK) - set(corrected_watch_df["tconst"])
)
assert not missing_verified_ok_tconsts, f"Verified-ok tconsts not found: {missing_verified_ok_tconsts}"

for tconst, verification in VERIFIED_DESCRIPTION_OK.items():
    row = corrected_watch_df.loc[corrected_watch_df["tconst"] == tconst].iloc[0]
    verified_ok_rows.append({
        "tconst": tconst,
        "primary_title": row["primary_title"],
        "release_year": row["release_year"],
        "current_description": row["description"],
        "source_url": verification["source_url"],
        "verification_note": verification["note"],
    })

mismatch_audit = final_df.copy()
mismatch_audit["title_description_overlap"] = mismatch_audit.apply(title_description_overlap, axis=1)
mismatch_audit["description_length"] = mismatch_audit["description"].astype(str).str.len()
mismatch_audit["title_fields_differ"] = mismatch_audit.apply(
    lambda row: len({
        str(row.get(column, ""))
        for column in ("primary_title", "display_title", "original_title")
        if pd.notna(row.get(column, ""))
    }) > 1,
    axis=1,
)

suspicious_description_mask = (
    (mismatch_audit["title_description_overlap"] == 0)
    & (mismatch_audit["num_votes"] >= 60_000)
    & (mismatch_audit["description_length"] <= 230)
) | mismatch_audit["tconst"].isin(DESCRIPTION_MISMATCH_CORRECTIONS)

needs_description_review = mismatch_audit[
    suspicious_description_mask
    & ~mismatch_audit["tconst"].isin(DESCRIPTION_MISMATCH_CORRECTIONS)
    & ~mismatch_audit["tconst"].isin(VERIFIED_DESCRIPTION_OK)
].copy()
needs_description_review["audit_reason"] = (
    "Heuristic flag only: high-vote row with short description and no overlap with "
    "title/original-title tokens. May be a false positive; verify externally before changing."
)

review_columns = [
    "tconst",
    "content_type",
    "primary_title",
    "display_title",
    "original_title",
    "release_year",
    "genres",
    "average_rating",
    "num_votes",
    "description",
    "title_description_overlap",
    "description_length",
    "title_fields_differ",
    "audit_reason",
]

corrected_description_sources = pd.DataFrame(source_rows)
verified_description_ok_sources = pd.DataFrame(verified_ok_rows)
corrected_watch_df.to_csv(CLEANED_PATH, index=False)
assert not corrected_watch_df["description"].isna().any(), "Corrected dataset has missing descriptions."
assert not corrected_watch_df["description"].astype(str).str.strip().eq("").any(), "Corrected dataset has empty descriptions."
assert not corrected_watch_df["tconst"].duplicated().any(), "Corrected dataset has duplicate tconst values."

identity_columns = ["primary_title", "display_title", "original_title", "release_year", "genres"]
for tconst in DESCRIPTION_MISMATCH_CORRECTIONS:
    before = final_df.loc[final_df["tconst"] == tconst, identity_columns].iloc[0]
    after = corrected_watch_df.loc[corrected_watch_df["tconst"] == tconst, identity_columns].iloc[0]
    assert before.equals(after), f"Identity columns changed for {tconst}."

print(f"Suspicious description rows found: {int(suspicious_description_mask.sum())}")
print(f"Corrected description rows: {len(DESCRIPTION_MISMATCH_CORRECTIONS)}")
print(f"Rows left for manual review: {len(needs_description_review)}")
print(f"Cleaned dataset updated with verified descriptions: {CLEANED_PATH}")

corrected_description_sources[[
    "tconst",
    "primary_title",
    "release_year",
    "corrected_description",
    "source_url",
]]


Suspicious description rows found: 413
Corrected description rows: 290
Rows left for manual review: 0
Cleaned dataset updated with verified descriptions: data/cleaned_watch_data.csv


,tconst,primary_title,release_year,corrected_description,source_url
0,tt7286456,Joker,2019,"In grim 1980s Gotham, isolated party clown Art...",https://en.wikipedia.org/wiki/Joker_(2019_film)
1,tt6751668,Parasite,2019,A poor Seoul family schemes its way into jobs ...,https://en.wikipedia.org/wiki/Parasite_(2019_f...
2,tt1663202,The Revenant,2015,After a bear mauling leaves frontiersman Hugh ...,https://en.wikipedia.org/wiki/The_Revenant_(20...
3,tt1392214,Prisoners,2013,"When two young girls vanish in Pennsylvania, a...",https://en.wikipedia.org/wiki/Prisoners_(2013_...
4,tt2096673,Inside Out,2015,"Inside the mind of young Riley, Joy, Sadness, ...",https://en.wikipedia.org/wiki/Inside_Out
...,...,...,...,...,...
285,tt12343534,Jujutsu Kaisen,2020,Yuji Itadori swallows a cursed object to save ...,https://en.wikipedia.org/wiki/Jujutsu_Kaisen_(...
286,tt12411074,Paranormal,2020,Skeptical hematologist Refaat Ismail is dragge...,https://en.wikipedia.org/wiki/Paranormal_(TV_s...
287,tt9612516,Space Force,2020,"A decorated general and a team of scientists, ...",https://en.wikipedia.org/wiki/Space_Force_(TV_...
288,tt0280249,Dragon Ball,1995,Monkey-tailed Goku meets Bulma and begins a co...,https://en.wikipedia.org/wiki/Dragon_Ball_(TV_...


## 11. Final Cleaning Checks

This final audit confirms that the cleaned dataset is ready for the enrichment and NLP notebooks. The important checks are row count, missing values, duplicate IMDb IDs, title/type coverage, and whether descriptions are strong enough to support semantic modeling.

Output from this notebook is not the deployed product by itself. It is the cleaned foundation used by the later notebooks that build richer descriptions, BERTopic labels, emotional facets, stress-test tables, and finally the backend catalog consumed by the website.


In [19]:
print("Final dataset shape:", watch_df.shape)
print("Total missing values:", int(watch_df.isna().sum().sum()))
print("Duplicate tconst values:", int(watch_df["tconst"].duplicated().sum()))
print("Missing descriptions:", int(watch_df["description"].isna().sum()))
print("Missing genres:", int(watch_df["genres"].isna().sum()))

watch_df.info()
watch_df.sample(10, random_state=42)


Final dataset shape: (7848, 12)
Total missing values: 0
Duplicate tconst values: 0
Missing descriptions: 0
Missing genres: 0
<class 'pandas.DataFrame'>
Index: 7848 entries, 0 to 7849
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             7848 non-null   str    
 1   content_type       7848 non-null   str    
 2   primary_title      7848 non-null   str    
 3   display_title      7848 non-null   str    
 4   original_title     7848 non-null   str    
 5   release_year       7848 non-null   int64  
 6   genres             7848 non-null   str    
 7   average_rating     7848 non-null   float64
 8   num_votes          7848 non-null   int64  
 9   description        7848 non-null   str    
 10  weighted_rating    7848 non-null   float64
 11  combined_features  7848 non-null   str    
dtypes: float64(2), int64(2), str(8)
memory usage: 1.0 MB


,tconst,content_type,primary_title,display_title,original_title,release_year,genres,average_rating,num_votes,description,weighted_rating,combined_features
4325,tt3973768,tvSeries,Hand of God,Hand of God,Hand of God,2014,"Crime,Drama",7.5,11777,Award-winning actor Ron Perlman stars as in th...,7.390,hand of god hand of god hand of god tvseries c...
2166,tt1194173,movie,The Bourne Legacy,The Bourne Legacy,The Bourne Legacy,2012,"Action,Adventure,Thriller",6.6,309253,When the actions of Jason Bourne spark a fire ...,6.719,the bourne legacy the bourne legacy the bourne...
4270,tt3775202,movie,I Am a Hero,I Am a Hero,Ai amu a hîrô,2015,"Action,Comedy,Horror",6.7,7380,A man witnesses a fatal traffic accident on hi...,7.291,i am a hero i am a hero ai amu a hîrô movie ac...
1905,tt11075264,movie,Thallumaala,Thallumaala,Thallumaala,2022,"Action,Comedy,Drama",7.0,5379,"Wazim meets his friends - Jamshi, Vikas and Ra...",7.336,thallumaala thallumaala thallumaala movie acti...
1454,tt0810788,tvSeries,Burn Notice,Burn Notice,Burn Notice,2007,"Action,Crime,Drama",8.0,76864,"Michael Westen is steamed -- or, more precisel...",7.732,burn notice burn notice burn notice tvseries a...
3307,tt1845307,tvSeries,2 Broke Girls,2 Broke Girls,2 Broke Girls,2011,Comedy,6.6,101509,Street-wise Max (Kat Dennings) doesn't expect ...,6.875,2 broke girls 2 broke girls 2 broke girls tvse...
4847,tt5705956,tvMiniSeries,The White Princess,The White Princess,The White Princess,2017,"Drama,History,Romance",7.5,14351,"In a tale of power, family, love and betrayal,...",7.394,the white princess the white princess the whit...
2464,tt13361448,tvSeries,Night Sky,Night Sky,Night Sky,2022,"Adventure,Drama,Fantasy",7.4,20625,Irene and Franklin York have kept secret a cha...,7.376,night sky night sky night sky tvseries adventu...
7716,tt6910006,movie,End of Sentence,End of Sentence,End of Sentence,2019,"Adventure,Drama",6.7,4329,"After being released from prison, a young man ...",7.320,end of sentence end of sentence end of sentenc...
6264,tt0308379,movie,Dark Water,Dark Water,Honogurai mizu no soko kara,2002,"Drama,Horror,Mystery",6.7,33750,Dahlia (Jennifer Connelly) wants to move away ...,7.118,dark water dark water honogurai mizu no soko k...
